In [1]:
# spark.stop()

In [24]:
import sys
import os
import json
import uuid

from pathlib import Path
from datetime import datetime, timezone

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window


# ============================================================
# PROJECT ROOT
#
# Notebook:
# capstone/product_analysis/notebooks/
#
# Project root:
# capstone/product_analysis/
# ============================================================

PROJECT_ROOT = Path.cwd().parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


# ============================================================
# APPLICATION DETAILS
# ============================================================

NOTEBOOK_NAME = "sales_order_lines_bronze_to_silver"
DATASET_NAME = "sales_order_lines"

CATALOG = "datalake"


# ============================================================
# BRONZE STREAM SOURCE
# ============================================================

BRONZE_PATH = (
    "hdfs:///capstone/warehouse/bronze/"
    "sales_order_lines"
)


# ============================================================
# STREAMING CHECKPOINT
# ============================================================

CHECKPOINT_PATH = (
    "hdfs://localhost:9000/"
    "capstone/checkpoints/"
    "bronze_to_silver/"
    "sales_order_lines"
)


# ============================================================
# ICEBERG WAREHOUSE
# ============================================================

WAREHOUSE = "hdfs:///capstone/warehouse/iceberg"


# ============================================================
# SILVER AND OPERATIONAL TABLES
# ============================================================

SILVER_TABLE = (
    f"{CATALOG}.silver.sales_order_lines"
)

REJECTED_TABLE = (
    f"{CATALOG}.quarantine."
    "sales_order_lines_rejected"
)

SILVER_AUDIT_TABLE = (
    f"{CATALOG}.operations."
    "silver_ingestion_audit"
)

EXPLORATION_TABLE = (
    f"{CATALOG}.operations.explorations"
)

EXPECTATION_TABLE = (
    f"{CATALOG}.operations."
    "expectation_profiles"
)


# ============================================================
# EXPECTED BRONZE SCHEMA FILE
# ============================================================

SCHEMA_FILE = (
    PROJECT_ROOT
    / "schemas"
    / "sales_order_lines_schema.json"
)


# ============================================================
# STREAMING AND DEDUPLICATION SETTINGS
# ============================================================

PRIMARY_KEY = "id"
LATEST_RECORD_COLUMN = "write_date"

STREAM_TRIGGER = "availableNow"


# ============================================================
# DISPLAY CONFIGURATION
# ============================================================

print("\nBronze-to-Silver configuration")
print("-" * 70)

print("Dataset:", DATASET_NAME)
print("Bronze source:", BRONZE_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

print("\nIceberg tables:")
print("Silver table:", SILVER_TABLE)
print("Rejected table:", REJECTED_TABLE)
print("Silver audit table:", SILVER_AUDIT_TABLE)
print("Exploration table:", EXPLORATION_TABLE)
print("Expectation table:", EXPECTATION_TABLE)

print("\nDeduplication:")
print("Primary key:", PRIMARY_KEY)
print("Latest-record column:", LATEST_RECORD_COLUMN)

print("\nSchema file:", SCHEMA_FILE)
print("Schema file exists:", SCHEMA_FILE.is_file())

Project root: /mnt/c/santhosh/capstone/product_analysis

Bronze-to-Silver configuration
----------------------------------------------------------------------
Dataset: sales_order_lines
Bronze source: hdfs:///capstone/warehouse/bronze/sales_order_lines
Checkpoint: hdfs://localhost:9000/capstone/checkpoints/bronze_to_silver/sales_order_lines

Iceberg tables:
Silver table: datalake.silver.sales_order_lines
Rejected table: datalake.quarantine.sales_order_lines_rejected
Silver audit table: datalake.operations.silver_ingestion_audit
Exploration table: datalake.operations.explorations
Expectation table: datalake.operations.expectation_profiles

Deduplication:
Primary key: id
Latest-record column: write_date

Schema file: /mnt/c/santhosh/capstone/product_analysis/schemas/sales_order_lines_schema.json
Schema file exists: True


In [3]:
# ============================================================
# CREATE SPARK SESSION
# ============================================================

builder = (
    SparkSession.builder
    .appName(NOTEBOOK_NAME)
    .master(
        os.environ.get(
            "SPARK_MASTER",
            "local[2]"
        )
    )

    # Iceberg SQL extensions required for MERGE INTO
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions."
        "IcebergSparkSessionExtensions"
    )

    # Hadoop-backed Iceberg catalog
    .config(
        f"spark.sql.catalog.{CATALOG}",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        f"spark.sql.catalog.{CATALOG}.type",
        "hadoop"
    )
    .config(
        f"spark.sql.catalog.{CATALOG}.warehouse",
        WAREHOUSE
    )

    # Consistent timestamp handling
    .config(
        "spark.sql.session.timeZone",
        "UTC"
    )

    # Preserve exact column-name case
    .config(
        "spark.sql.caseSensitive",
        "true"
    )

    # Suitable for the current local development environment
    .config(
        "spark.sql.shuffle.partitions",
        "2"
    )

    # Prevent accidental overwrite of unrelated partitions
    .config(
        "spark.sql.sources.partitionOverwriteMode",
        "dynamic"
    )
)


# ============================================================
# OPTIONAL ICEBERG RUNTIME JAR
# ============================================================

iceberg_runtime_jar = os.environ.get(
    "ICEBERG_SPARK_RUNTIME_JAR"
)

if iceberg_runtime_jar:
    builder = builder.config(
        "spark.jars",
        iceberg_runtime_jar
    )

    print(
        "Using Iceberg runtime JAR:",
        iceberg_runtime_jar
    )
else:
    print(
        "ICEBERG_SPARK_RUNTIME_JAR is not set. "
        "Assuming Iceberg dependencies are already "
        "available to Spark."
    )


# ============================================================
# START SPARK
# ============================================================

spark = builder.getOrCreate()

spark.sparkContext.setLogLevel("WARN")


# ============================================================
# VERIFY SPARK CONFIGURATION
# ============================================================

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

print(
    "Iceberg catalog:",
    spark.conf.get(
        f"spark.sql.catalog.{CATALOG}"
    )
)

print(
    "Iceberg catalog type:",
    spark.conf.get(
        f"spark.sql.catalog.{CATALOG}.type"
    )
)

print(
    "Iceberg warehouse:",
    spark.conf.get(
        f"spark.sql.catalog.{CATALOG}.warehouse"
    )
)


ICEBERG_SPARK_RUNTIME_JAR is not set. Assuming Iceberg dependencies are already available to Spark.


26/10/12 03:56:21 WARN Utils: Your hostname, 5CD6260W0K resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/12 03:56:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/12 03:56:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.9
Spark master: local[2]
Iceberg catalog: org.apache.iceberg.spark.SparkCatalog
Iceberg catalog type: hadoop
Iceberg warehouse: hdfs:///capstone/warehouse/iceberg


In [4]:
# ============================================================
# LOAD SALES ORDER LINES SCHEMA JSON
# ============================================================

if not SCHEMA_FILE.is_file():
    raise FileNotFoundError(
        f"Schema file does not exist: {SCHEMA_FILE}"
    )

with open(
    SCHEMA_FILE,
    mode="r",
    encoding="utf-8"
) as schema_file:
    schema_definition = json.load(schema_file)

print("Loaded schema definition:")
print(
    json.dumps(
        schema_definition,
        indent=4
    )
)


# ============================================================
# MAP JSON TYPE NAMES TO SPARK DATA TYPES
# ============================================================

SPARK_TYPE_MAPPING = {
    "string": T.StringType(),
    "integer": T.IntegerType(),
    "int": T.IntegerType(),
    "long": T.LongType(),
    "double": T.DoubleType(),
    "float": T.FloatType(),
    "boolean": T.BooleanType(),
    "date": T.DateType(),
    "timestamp": T.TimestampType(),
}


# ============================================================
# BUILD BUSINESS SCHEMA
# ============================================================

business_schema_fields = []

for column_name, datatype_name in schema_definition.items():
    normalized_type = datatype_name.strip().lower()

    if normalized_type not in SPARK_TYPE_MAPPING:
        raise ValueError(
            f"Unsupported datatype '{datatype_name}' "
            f"for column '{column_name}'"
        )

    business_schema_fields.append(
        T.StructField(
            column_name,
            SPARK_TYPE_MAPPING[normalized_type],
            True
        )
    )

business_schema = T.StructType(
    business_schema_fields
)


# ============================================================
# ADD BRONZE PARTITION COLUMNS
#
# The Bronze path structure contains:
#
# batch_id=api-00002/
# file_00000/
# write_date_partition=2026-10-07/
#
# Spark discovers Hive-style key=value directories as columns
# when reading from their common root.
# ============================================================

bronze_stream_schema = T.StructType(
    business_schema.fields
    + [
        T.StructField(
            "batch_id",
            T.StringType(),
            True
        ),
        T.StructField(
            "write_date_partition",
            T.DateType(),
            True
        ),
    ]
)


# ============================================================
# DISPLAY BUSINESS SCHEMA
# ============================================================

print("\nBusiness schema:")
print(
    business_schema.simpleString()
)

print("\nBusiness schema fields:")

for field in business_schema.fields:
    print(
        f"{field.name:<25}"
        f"{field.dataType.simpleString():<15}"
        f"nullable={field.nullable}"
    )


# ============================================================
# DISPLAY COMPLETE BRONZE STREAM SCHEMA
# ============================================================

print("\nComplete Bronze stream schema:")
print(
    bronze_stream_schema.simpleString()
)

print("\nBronze stream fields:")

for field in bronze_stream_schema.fields:
    print(
        f"{field.name:<25}"
        f"{field.dataType.simpleString():<15}"
        f"nullable={field.nullable}"
    )


# ============================================================
# BASIC SCHEMA VALIDATION
# ============================================================

required_source_columns = {
    "id",
    "order_id",
    "product_id",
    "product_uom_qty",
    "price_unit",
    "discount",
    "price_subtotal",
    "create_date",
    "write_date",
    "qty_delivered",
    "qty_invoiced",
}

business_schema_columns = {
    field.name
    for field in business_schema.fields
}

missing_required_columns = (
    required_source_columns
    - business_schema_columns
)

if missing_required_columns:
    raise ValueError(
        "Required columns are missing from the schema: "
        f"{sorted(missing_required_columns)}"
    )

print(
    "\nRequired-column validation: PASSED"
)

Loaded schema definition:
{
    "id": "long",
    "order_id": "string",
    "product_id": "string",
    "product_uom_qty": "double",
    "price_unit": "double",
    "discount": "double",
    "price_subtotal": "double",
    "create_date": "timestamp",
    "write_date": "timestamp",
    "qty_delivered": "double",
    "qty_invoiced": "double"
}

Business schema:
struct<id:bigint,order_id:string,product_id:string,product_uom_qty:double,price_unit:double,discount:double,price_subtotal:double,create_date:timestamp,write_date:timestamp,qty_delivered:double,qty_invoiced:double>

Business schema fields:
id                       bigint         nullable=True
order_id                 string         nullable=True
product_id               string         nullable=True
product_uom_qty          double         nullable=True
price_unit               double         nullable=True
discount                 double         nullable=True
price_subtotal           double         nullable=True
create_date         

In [5]:
# ============================================================
# STATIC BRONZE READ
#
# This is a validation read before creating the stream.
# Only the business schema is passed because CSV files contain
# the 11 business columns. Partition metadata is derived from
# the physical HDFS file path.
# ============================================================

bronze_static_raw_df = (
    spark.read
    .format("csv")
    .schema(business_schema)
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .option("encoding", "UTF-8")
    .option(
        "timestampFormat",
        "yyyy-MM-dd HH:mm:ss"
    )
    .option(
        "recursiveFileLookup",
        "true"
    )
    .load(BRONZE_PATH)
)


# ============================================================
# ADD SOURCE FILE AND PARTITION METADATA
#
# Example input path:
#
# hdfs:///capstone/warehouse/bronze/sales_order_lines/
# batch_id=api-00002/file_00000/
# write_date_partition=2026-10-07/part-....csv
# ============================================================

bronze_static_df = (
    bronze_static_raw_df

    # Preserve the exact Bronze source file for lineage
    .withColumn(
        "_bronze_source_file",
        F.input_file_name()
    )

    # Extract batch ID from the HDFS path
    .withColumn(
        "batch_id",
        F.regexp_extract(
            F.col("_bronze_source_file"),
            r"batch_id=([^/]+)",
            1
        )
    )

    # Extract partition date from the HDFS path
    .withColumn(
        "write_date_partition",
        F.to_date(
            F.regexp_extract(
                F.col("_bronze_source_file"),
                r"write_date_partition=([^/]+)",
                1
            ),
            "yyyy-MM-dd"
        )
    )
)


# ============================================================
# MATERIALIZE AND DISPLAY BASIC INFORMATION
# ============================================================

bronze_static_count = bronze_static_df.count()

print("Static Bronze record count:", bronze_static_count)

print("\nStatic Bronze schema:")
bronze_static_df.printSchema()

print("\nStatic Bronze preview:")

bronze_static_df.select(
    "id",
    "order_id",
    "product_id",
    "write_date",
    "batch_id",
    "write_date_partition",
    "_bronze_source_file"
).show(
    10,
    truncate=False
)


# ============================================================
# DISPLAY DISCOVERED BATCHES
# ============================================================

print("\nDiscovered Bronze batches:")

bronze_static_df.groupBy(
    "batch_id"
).agg(
    F.count("*").alias("record_count"),
    F.min("write_date").alias("minimum_write_date"),
    F.max("write_date").alias("maximum_write_date"),
    F.countDistinct(
        "_bronze_source_file"
    ).alias("source_file_count")
).orderBy(
    "batch_id"
).show(
    truncate=False
)


# ============================================================
# DISPLAY DISCOVERED DATE PARTITIONS
# ============================================================

print("\nDiscovered write-date partitions:")

bronze_static_df.groupBy(
    "batch_id",
    "write_date_partition"
).count().orderBy(
    "batch_id",
    "write_date_partition"
).show(
    100,
    truncate=False
)


# ============================================================
# VALIDATE PATH-DERIVED METADATA
# ============================================================

missing_batch_id_count = (
    bronze_static_df
    .filter(
        F.col("batch_id").isNull()
        | (F.trim(F.col("batch_id")) == "")
    )
    .count()
)

missing_partition_date_count = (
    bronze_static_df
    .filter(
        F.col("write_date_partition").isNull()
    )
    .count()
)

print("\nMetadata validation")
print("-" * 70)

print(
    "Records with missing batch_id:",
    missing_batch_id_count
)

print(
    "Records with missing write_date_partition:",
    missing_partition_date_count
)

if missing_batch_id_count > 0:
    raise ValueError(
        f"{missing_batch_id_count} Bronze records do not "
        "contain batch_id in their source path."
    )

if missing_partition_date_count > 0:
    raise ValueError(
        f"{missing_partition_date_count} Bronze records do not "
        "contain a valid write_date_partition in their path."
    )

print(
    "Bronze path-metadata validation: PASSED"
)


# ============================================================
# COMPARE WRITE DATE WITH PARTITION DATE
# ============================================================

partition_date_mismatch_count = (
    bronze_static_df
    .filter(
        F.to_date(F.col("write_date"))
        != F.col("write_date_partition")
    )
    .count()
)

print(
    "write_date/partition-date mismatches:",
    partition_date_mismatch_count
)

if partition_date_mismatch_count == 0:
    print(
        "Write-date partition validation: PASSED"
    )
else:
    print(
        "Write-date partition validation: WARNING"
    )

Static Bronze record count: 10497

Static Bronze schema:
root
 |-- id: long (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_uom_qty: double (nullable = true)
 |-- price_unit: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- price_subtotal: double (nullable = true)
 |-- create_date: timestamp (nullable = true)
 |-- write_date: timestamp (nullable = true)
 |-- qty_delivered: double (nullable = true)
 |-- qty_invoiced: double (nullable = true)
 |-- _bronze_source_file: string (nullable = false)
 |-- batch_id: string (nullable = false)
 |-- write_date_partition: date (nullable = true)


Static Bronze preview:
+---+--------------+------------------------------------------------------+----------+---------+--------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
# ============================================================
# INSPECT BRONZE VALUES WITHOUT TIMESTAMP CONVERSION
#
# Read every CSV field as STRING so that we can see the exact
# timestamp text stored in HDFS.
# ============================================================

bronze_string_schema = T.StructType(
    [
        T.StructField(
            field.name,
            T.StringType(),
            True,
        )
        for field in business_schema.fields
    ]
)

bronze_raw_strings_df = (
    spark.read
    .format("csv")
    .schema(bronze_string_schema)
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .option("encoding", "UTF-8")
    .option("recursiveFileLookup", "true")
    .load(BRONZE_PATH)
    .withColumn(
        "_bronze_source_file",
        F.input_file_name()
    )
    .withColumn(
        "batch_id",
        F.regexp_extract(
            F.col("_bronze_source_file"),
            r"batch_id=([^/]+)",
            1,
        )
    )
)

# ============================================================
# DISPLAY DISTINCT RAW TIMESTAMP VALUES
# ============================================================

print("Raw timestamp values stored in Bronze:")

bronze_raw_strings_df.select(
    "batch_id",
    "create_date",
    "write_date",
    "_bronze_source_file",
).where(
    F.col("batch_id").isin(
        "api-00002",
        "api-00002-rerun-01",
        "api-00002-rerun-02",
    )
).show(
    5,
    truncate=False,
)

# ============================================================
# DISPLAY TIMESTAMP FORMAT PATTERNS
# ============================================================

timestamp_pattern_df = (
    bronze_raw_strings_df
    .select(
        "batch_id",
        "create_date",
        "write_date",
    )
    .dropDuplicates(
        [
            "batch_id",
            "create_date",
            "write_date",
        ]
    )
)

print("\nSample distinct timestamp representations:")

timestamp_pattern_df.orderBy(
    "batch_id"
).show(
    5,
    truncate=False,
)

Raw timestamp values stored in Bronze:
+---------+------------------------+------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|batch_id |create_date             |write_date              |_bronze_source_file                                                                                                                                                                     |
+---------+------------------------+------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|api-00002|2026-10-07T17:50:27.000Z|2026-10-07T17:50:39.000Z|hdfs://localhost:9000/capstone/warehouse/bronze/sales_order_lines/batch_id=api-00002/file_00000/write_date_partition=2026-10-07/part-00000-1f748cd3-65b6-43f6-9

In [7]:
# ============================================================
# BRONZE TIMESTAMP FORMAT
#
# Actual Bronze representation:
# 2026-10-07T17:50:27.000Z
# ============================================================

BRONZE_TIMESTAMP_FORMAT = (
    "yyyy-MM-dd'T'HH:mm:ss.SSSX"
)


# ============================================================
# RE-READ BRONZE WITH CORRECT TIMESTAMP FORMAT
# ============================================================

bronze_static_raw_df = (
    spark.read
    .format("csv")
    .schema(business_schema)
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .option("encoding", "UTF-8")
    .option(
        "timestampFormat",
        BRONZE_TIMESTAMP_FORMAT
    )
    .option(
        "recursiveFileLookup",
        "true"
    )
    .load(BRONZE_PATH)
)


# ============================================================
# ADD SOURCE LINEAGE AND PATH METADATA
# ============================================================

bronze_static_df = (
    bronze_static_raw_df

    .withColumn(
        "_bronze_source_file",
        F.input_file_name()
    )

    .withColumn(
        "batch_id",
        F.regexp_extract(
            F.col("_bronze_source_file"),
            r"batch_id=([^/]+)",
            1
        )
    )

    .withColumn(
        "write_date_partition",
        F.to_date(
            F.regexp_extract(
                F.col("_bronze_source_file"),
                r"write_date_partition=([^/]+)",
                1
            ),
            "yyyy-MM-dd"
        )
    )
)


# ============================================================
# VALIDATE TIMESTAMP PARSING
# ============================================================

timestamp_validation_df = (
    bronze_static_df
    .groupBy("batch_id")
    .agg(
        F.count("*").alias(
            "record_count"
        ),

        F.sum(
            F.when(
                F.col("create_date").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_create_date_count"
        ),

        F.sum(
            F.when(
                F.col("write_date").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_write_date_count"
        ),

        F.min(
            "write_date"
        ).alias(
            "minimum_write_date"
        ),

        F.max(
            "write_date"
        ).alias(
            "maximum_write_date"
        ),
    )
    .orderBy("batch_id")
)

print(
    "Bronze timestamp parsing validation:"
)

timestamp_validation_df.show(
    truncate=False
)


# ============================================================
# VALIDATE NORMAL PRODUCTION BATCHES
#
# Malformed test batches are intentionally excluded from this
# validation because they may contain rejected test values.
# ============================================================

PRODUCTION_BATCHES = [
    "api-00002",
    "api-00002-rerun-01",
    "api-00002-rerun-02",
]

production_timestamp_failures = (
    timestamp_validation_df
    .filter(
        F.col("batch_id").isin(
            PRODUCTION_BATCHES
        )
    )
    .filter(
        (F.col("null_create_date_count") > 0)
        | (F.col("null_write_date_count") > 0)
    )
    .count()
)

if production_timestamp_failures > 0:
    raise ValueError(
        "Timestamp parsing failed for one or more "
        "production Bronze batches."
    )

print(
    "Production timestamp validation: PASSED"
)


# ============================================================
# VERIFY TIMESTAMP/PARTITION CONSISTENCY
# ============================================================

partition_mismatch_count = (
    bronze_static_df
    .filter(
        F.col("write_date").isNotNull()
        & F.col(
            "write_date_partition"
        ).isNotNull()
        & (
            F.to_date(
                F.col("write_date")
            )
            != F.col(
                "write_date_partition"
            )
        )
    )
    .count()
)

print(
    "Timestamp/partition mismatches:",
    partition_mismatch_count
)

if partition_mismatch_count == 0:
    print(
        "Timestamp/partition validation: PASSED"
    )
else:
    print(
        "Timestamp/partition validation: WARNING"
    )


# ============================================================
# DISPLAY CORRECTED SAMPLE
# ============================================================

print("\nCorrectly parsed Bronze sample:")

bronze_static_df.select(
    "id",
    "create_date",
    "write_date",
    "write_date_partition",
    "batch_id"
).filter(
    F.col("batch_id") == "api-00002"
).show(
    10,
    truncate=False
)

Bronze timestamp parsing validation:
+----------------------+------------+----------------------+---------------------+-------------------+-------------------+
|batch_id              |record_count|null_create_date_count|null_write_date_count|minimum_write_date |maximum_write_date |
+----------------------+------------+----------------------+---------------------+-------------------+-------------------+
|api-00002             |3497        |0                     |0                    |2026-10-07 17:50:29|2026-10-07 17:54:11|
|api-00002-malformed-02|3           |3                     |3                    |NULL               |NULL               |
|api-00002-malformed-03|3           |3                     |3                    |NULL               |NULL               |
|api-00002-rerun-01    |3497        |0                     |0                    |2026-10-07 17:50:29|2026-10-07 17:54:11|
|api-00002-rerun-02    |3497        |0                     |0                    |2026-10-07 17:50:29|

In [8]:
# ============================================================
# IMPORT DATA PROFILING FUNCTIONS
# ============================================================

import inspect
import importlib

import data_profilers.exploration as exploration_module
import data_profilers.great_expectations as gx_profiler_module


# Reload modules so notebook uses the latest saved code
exploration_module = importlib.reload(
    exploration_module
)

gx_profiler_module = importlib.reload(
    gx_profiler_module
)


# Assign profiler functions
explore_data = (
    exploration_module.explore_data
)

write_exploration_to_iceberg = (
    exploration_module.write_exploration_to_iceberg
)

profile_expectations = (
    gx_profiler_module.profile_expectations
)


# ============================================================
# VERIFY MODULE LOCATIONS
# ============================================================

print("Profiler module locations")
print("-" * 70)

print(
    "Exploration module:",
    exploration_module.__file__
)

print(
    "Great Expectations profiler module:",
    gx_profiler_module.__file__
)


# ============================================================
# VERIFY EXPECTED PROJECT LOCATIONS
# ============================================================

expected_exploration_file = (
    PROJECT_ROOT
    / "data_profilers"
    / "exploration.py"
).resolve()

expected_gx_profiler_file = (
    PROJECT_ROOT
    / "data_profilers"
    / "great_expectations.py"
).resolve()

actual_exploration_file = Path(
    exploration_module.__file__
).resolve()

actual_gx_profiler_file = Path(
    gx_profiler_module.__file__
).resolve()


if actual_exploration_file != expected_exploration_file:
    raise ImportError(
        "Incorrect exploration module loaded.\n"
        f"Expected: {expected_exploration_file}\n"
        f"Actual:   {actual_exploration_file}"
    )

if actual_gx_profiler_file != expected_gx_profiler_file:
    raise ImportError(
        "Incorrect Great Expectations profiler "
        "module loaded.\n"
        f"Expected: {expected_gx_profiler_file}\n"
        f"Actual:   {actual_gx_profiler_file}"
    )

print(
    "\nProfiler location validation: PASSED"
)


# ============================================================
# VERIFY FUNCTION SIGNATURES
# ============================================================

print("\nProfiler function signatures")
print("-" * 70)

print(
    "explore_data:",
    inspect.signature(explore_data)
)

print(
    "\nwrite_exploration_to_iceberg:",
    inspect.signature(
        write_exploration_to_iceberg
    )
)

print(
    "\nprofile_expectations:",
    inspect.signature(
        profile_expectations
    )
)


# ============================================================
# VERIFY GREAT EXPECTATIONS INSTALLATION
# ============================================================

try:
    import great_expectations as gx

    gx_version = getattr(
        gx,
        "__version__",
        "version unavailable"
    )

    print(
        "\nGreat Expectations version:",
        gx_version
    )

except ImportError as error:
    print("error")


Profiler module locations
----------------------------------------------------------------------
Exploration module: /mnt/c/santhosh/capstone/product_analysis/data_profilers/exploration.py
Great Expectations profiler module: /mnt/c/santhosh/capstone/product_analysis/data_profilers/great_expectations.py

Profiler location validation: PASSED

Profiler function signatures
----------------------------------------------------------------------
explore_data: (spark, dataframe_or_table, dataset_name=None, run_id=None, batch_id=None, profile_scope='BATCH', processing_stage='LANDING', profile_level='FULL', required_columns=None, source_file_column=None, columns=None, approximate_distinct=True, relative_sd=0.05)

write_exploration_to_iceberg: (spark, exploration_results, catalog_name='datalake', namespace_name='operations', table_name='explorations')

profile_expectations: (df, expectations, *, dataset_name=None, run_id=None, audit_table='datalake.operations.expectation_profiles')

Great Expecta

In [9]:
# ============================================================
# GREAT EXPECTATIONS MONITORING RULES
#
# These expectations provide batch-level quality monitoring.
# Equivalent Spark conditions will later perform row-level
# filtering and route invalid records to the rejected table.
# ============================================================

gx_expectations = [

    # --------------------------------------------------------
    # PRIMARY KEY
    # --------------------------------------------------------

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="id"
    ),

    # --------------------------------------------------------
    # ODOO RELATION COLUMNS
    # --------------------------------------------------------

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="order_id"
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="product_id"
    ),

    # --------------------------------------------------------
    # QUANTITY AND PRICE COLUMNS
    # --------------------------------------------------------

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="product_uom_qty"
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="price_unit"
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="price_subtotal"
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="qty_delivered"
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="qty_invoiced"
    ),

    # --------------------------------------------------------
    # DISCOUNT VALIDATION
    # --------------------------------------------------------

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="discount"
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="discount",
        min_value=0.0,
        max_value=100.0
    ),

    # --------------------------------------------------------
    # TIMESTAMP COLUMNS
    # --------------------------------------------------------

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="create_date"
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="write_date"
    ),
]


# ============================================================
# DISPLAY CONFIGURED EXPECTATIONS
# ============================================================

print(
    "Configured Great Expectations rules:",
    len(gx_expectations)
)

print("\nExpectation rules")
print("-" * 70)

for rule_number, expectation in enumerate(
    gx_expectations,
    start=1
):
    expectation_type = (
        expectation.__class__.__name__
    )

    expectation_column = getattr(
        expectation,
        "column",
        None
    )

    print(
        f"{rule_number:>2}. "
        f"{expectation_type:<45} "
        f"column={expectation_column}"
    )

Configured Great Expectations rules: 12

Expectation rules
----------------------------------------------------------------------
 1. ExpectColumnValuesToNotBeNull                 column=id
 2. ExpectColumnValuesToNotBeNull                 column=order_id
 3. ExpectColumnValuesToNotBeNull                 column=product_id
 4. ExpectColumnValuesToNotBeNull                 column=product_uom_qty
 5. ExpectColumnValuesToNotBeNull                 column=price_unit
 6. ExpectColumnValuesToNotBeNull                 column=price_subtotal
 7. ExpectColumnValuesToNotBeNull                 column=qty_delivered
 8. ExpectColumnValuesToNotBeNull                 column=qty_invoiced
 9. ExpectColumnValuesToNotBeNull                 column=discount
10. ExpectColumnValuesToBeBetween                 column=discount
11. ExpectColumnValuesToNotBeNull                 column=create_date
12. ExpectColumnValuesToNotBeNull                 column=write_date


In [10]:
# ============================================================
# SILVER TABLE SCHEMA
#
# Raw Odoo relation strings are not carried into Silver.
# order_id and product_id contain the parsed numeric IDs.
# ============================================================

silver_schema = T.StructType([
    T.StructField(
        "sales_order_line_id",
        T.LongType(),
        False
    ),
    T.StructField(
        "sales_order_id",
        T.LongType(),
        False
    ),
    T.StructField(
        "sales_order_name",
        T.StringType(),
        True
    ),
    T.StructField(
        "product_id",
        T.LongType(),
        False
    ),
    T.StructField(
        "product_display_name",
        T.StringType(),
        True
    ),
    T.StructField(
        "product_uom_qty",
        T.DoubleType(),
        False
    ),
    T.StructField(
        "price_unit",
        T.DoubleType(),
        False
    ),
    T.StructField(
        "discount",
        T.DoubleType(),
        False
    ),
    T.StructField(
        "price_subtotal",
        T.DoubleType(),
        False
    ),
    T.StructField(
        "create_date",
        T.TimestampType(),
        False
    ),
    T.StructField(
        "write_date",
        T.TimestampType(),
        False
    ),
    T.StructField(
        "qty_delivered",
        T.DoubleType(),
        False
    ),
    T.StructField(
        "qty_invoiced",
        T.DoubleType(),
        False
    ),

    # Source lineage
    T.StructField(
        "source_batch_id",
        T.StringType(),
        False
    ),
    T.StructField(
        "source_file",
        T.StringType(),
        False
    ),

    # Streaming lineage
    T.StructField(
        "stream_batch_id",
        T.LongType(),
        False
    ),
    T.StructField(
        "pipeline_run_id",
        T.StringType(),
        False
    ),

    # Silver processing metadata
    T.StructField(
        "silver_created_at",
        T.TimestampType(),
        False
    ),
    T.StructField(
        "silver_updated_at",
        T.TimestampType(),
        False
    ),
])


# ============================================================
# REJECTED-RECORD TABLE SCHEMA
#
# Rejected records retain the original Bronze values as well
# as any successfully parsed Odoo relation values.
# ============================================================

rejected_schema = T.StructType([
    # Original Bronze columns
    T.StructField(
        "id",
        T.LongType(),
        True
    ),
    T.StructField(
        "order_id_raw",
        T.StringType(),
        True
    ),
    T.StructField(
        "product_id_raw",
        T.StringType(),
        True
    ),
    T.StructField(
        "product_uom_qty",
        T.DoubleType(),
        True
    ),
    T.StructField(
        "price_unit",
        T.DoubleType(),
        True
    ),
    T.StructField(
        "discount",
        T.DoubleType(),
        True
    ),
    T.StructField(
        "price_subtotal",
        T.DoubleType(),
        True
    ),
    T.StructField(
        "create_date",
        T.TimestampType(),
        True
    ),
    T.StructField(
        "write_date",
        T.TimestampType(),
        True
    ),
    T.StructField(
        "qty_delivered",
        T.DoubleType(),
        True
    ),
    T.StructField(
        "qty_invoiced",
        T.DoubleType(),
        True
    ),

    # Parsed Odoo relationship values
    T.StructField(
        "parsed_sales_order_id",
        T.LongType(),
        True
    ),
    T.StructField(
        "parsed_sales_order_name",
        T.StringType(),
        True
    ),
    T.StructField(
        "parsed_product_id",
        T.LongType(),
        True
    ),
    T.StructField(
        "parsed_product_display_name",
        T.StringType(),
        True
    ),

    # Validation results
    T.StructField(
        "rejection_reasons",
        T.ArrayType(
            T.StringType(),
            containsNull=False
        ),
        False
    ),
    T.StructField(
        "failed_rule_names",
        T.ArrayType(
            T.StringType(),
            containsNull=False
        ),
        False
    ),

    # Source lineage
    T.StructField(
        "source_batch_id",
        T.StringType(),
        True
    ),
    T.StructField(
        "source_file",
        T.StringType(),
        True
    ),

    # Streaming lineage
    T.StructField(
        "stream_batch_id",
        T.LongType(),
        False
    ),
    T.StructField(
        "pipeline_run_id",
        T.StringType(),
        False
    ),
    T.StructField(
        "rejected_at",
        T.TimestampType(),
        False
    ),
])


# ============================================================
# SILVER INGESTION AUDIT SCHEMA
#
# One summary record will be written for every micro-batch.
# ============================================================

silver_audit_schema = T.StructType([
    T.StructField(
        "audit_id",
        T.StringType(),
        False
    ),
    T.StructField(
        "pipeline_run_id",
        T.StringType(),
        False
    ),
    T.StructField(
        "stream_batch_id",
        T.LongType(),
        False
    ),
    T.StructField(
        "dataset_name",
        T.StringType(),
        False
    ),
    T.StructField(
        "processing_stage",
        T.StringType(),
        False
    ),
    T.StructField(
        "status",
        T.StringType(),
        False
    ),

    # Batch counts
    T.StructField(
        "source_count",
        T.LongType(),
        False
    ),
    T.StructField(
        "valid_before_dedup_count",
        T.LongType(),
        False
    ),
    T.StructField(
        "rejected_count",
        T.LongType(),
        False
    ),
    T.StructField(
        "duplicate_count",
        T.LongType(),
        False
    ),
    T.StructField(
        "valid_after_dedup_count",
        T.LongType(),
        False
    ),

    # Merge verification
    T.StructField(
        "incoming_keys_found_in_silver",
        T.LongType(),
        True
    ),
    T.StructField(
        "reconciliation_status",
        T.StringType(),
        False
    ),

    # Profiler/checkpoint outcomes
    T.StructField(
        "exploration_status",
        T.StringType(),
        True
    ),
    T.StructField(
        "expectation_status",
        T.StringType(),
        True
    ),

    # Error handling
    T.StructField(
        "error_message",
        T.StringType(),
        True
    ),

    # Processing timestamps
    T.StructField(
        "started_at",
        T.TimestampType(),
        False
    ),
    T.StructField(
        "completed_at",
        T.TimestampType(),
        True
    ),
])


# ============================================================
# DISPLAY DEFINED SCHEMAS
# ============================================================

print("Silver schema")
print("-" * 70)
print(silver_schema.simpleString())

print("\nRejected-record schema")
print("-" * 70)
print(rejected_schema.simpleString())

print("\nSilver audit schema")
print("-" * 70)
print(silver_audit_schema.simpleString())


# ============================================================
# BASIC SCHEMA VALIDATION
# ============================================================

silver_column_names = [
    field.name
    for field in silver_schema.fields
]

rejected_column_names = [
    field.name
    for field in rejected_schema.fields
]

audit_column_names = [
    field.name
    for field in silver_audit_schema.fields
]


assert len(silver_column_names) == len(
    set(silver_column_names)
), "Duplicate columns found in Silver schema."

assert len(rejected_column_names) == len(
    set(rejected_column_names)
), "Duplicate columns found in rejected schema."

assert len(audit_column_names) == len(
    set(audit_column_names)
), "Duplicate columns found in audit schema."


required_silver_columns = {
    "sales_order_line_id",
    "sales_order_id",
    "product_id",
    "write_date",
    "source_batch_id",
    "stream_batch_id",
    "pipeline_run_id",
}

missing_silver_columns = (
    required_silver_columns
    - set(silver_column_names)
)

if missing_silver_columns:
    raise ValueError(
        "Required Silver columns are missing: "
        f"{sorted(missing_silver_columns)}"
    )


print(
    "\nSchema validation: PASSED"
)

Silver schema
----------------------------------------------------------------------
struct<sales_order_line_id:bigint,sales_order_id:bigint,sales_order_name:string,product_id:bigint,product_display_name:string,product_uom_qty:double,price_unit:double,discount:double,price_subtotal:double,create_date:timestamp,write_date:timestamp,qty_delivered:double,qty_invoiced:double,source_batch_id:string,source_file:string,stream_batch_id:bigint,pipeline_run_id:string,silver_created_at:timestamp,silver_updated_at:timestamp>

Rejected-record schema
----------------------------------------------------------------------
struct<id:bigint,order_id_raw:string,product_id_raw:string,product_uom_qty:double,price_unit:double,discount:double,price_subtotal:double,create_date:timestamp,write_date:timestamp,qty_delivered:double,qty_invoiced:double,parsed_sales_order_id:bigint,parsed_sales_order_name:string,parsed_product_id:bigint,parsed_product_display_name:string,rejection_reasons:array<string>,failed_rule_

In [11]:
# ============================================================
# CREATE ICEBERG NAMESPACES
# ============================================================

required_namespaces = [
    "silver",
    "quarantine",
    "operations",
]

for namespace in required_namespaces:
    spark.sql(
        f"""
        CREATE NAMESPACE IF NOT EXISTS
        {CATALOG}.{namespace}
        """
    )

    print(
        f"Namespace ready: "
        f"{CATALOG}.{namespace}"
    )


# ============================================================
# CREATE SILVER TABLE
#
# Partitioning by days(write_date) is suitable for time-based
# filtering while avoiding one partition per full timestamp.
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE}
    (
        sales_order_line_id BIGINT NOT NULL,
        sales_order_id BIGINT NOT NULL,
        sales_order_name STRING,
        product_id BIGINT NOT NULL,
        product_display_name STRING,

        product_uom_qty DOUBLE NOT NULL,
        price_unit DOUBLE NOT NULL,
        discount DOUBLE NOT NULL,
        price_subtotal DOUBLE NOT NULL,

        create_date TIMESTAMP NOT NULL,
        write_date TIMESTAMP NOT NULL,

        qty_delivered DOUBLE NOT NULL,
        qty_invoiced DOUBLE NOT NULL,

        source_batch_id STRING NOT NULL,
        source_file STRING NOT NULL,

        stream_batch_id BIGINT NOT NULL,
        pipeline_run_id STRING NOT NULL,

        silver_created_at TIMESTAMP NOT NULL,
        silver_updated_at TIMESTAMP NOT NULL
    )
    USING iceberg
    PARTITIONED BY (
        days(write_date)
    )
    TBLPROPERTIES (
        'format-version' = '2',
        'write.format.default' = 'parquet'
    )
    """
)

print(
    f"Silver table ready: {SILVER_TABLE}"
)


# ============================================================
# CREATE REJECTED-RECORD TABLE
#
# Rejected records are partitioned by rejection date.
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {REJECTED_TABLE}
    (
        id BIGINT,
        order_id_raw STRING,
        product_id_raw STRING,

        product_uom_qty DOUBLE,
        price_unit DOUBLE,
        discount DOUBLE,
        price_subtotal DOUBLE,

        create_date TIMESTAMP,
        write_date TIMESTAMP,

        qty_delivered DOUBLE,
        qty_invoiced DOUBLE,

        parsed_sales_order_id BIGINT,
        parsed_sales_order_name STRING,
        parsed_product_id BIGINT,
        parsed_product_display_name STRING,

        rejection_reasons ARRAY<STRING> NOT NULL,
        failed_rule_names ARRAY<STRING> NOT NULL,

        source_batch_id STRING,
        source_file STRING,

        stream_batch_id BIGINT NOT NULL,
        pipeline_run_id STRING NOT NULL,
        rejected_at TIMESTAMP NOT NULL
    )
    USING iceberg
    PARTITIONED BY (
        days(rejected_at)
    )
    TBLPROPERTIES (
        'format-version' = '2',
        'write.format.default' = 'parquet'
    )
    """
)

print(
    f"Rejected table ready: {REJECTED_TABLE}"
)


# ============================================================
# CREATE SILVER INGESTION AUDIT TABLE
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {SILVER_AUDIT_TABLE}
    (
        audit_id STRING NOT NULL,
        pipeline_run_id STRING NOT NULL,
        stream_batch_id BIGINT NOT NULL,

        dataset_name STRING NOT NULL,
        processing_stage STRING NOT NULL,
        status STRING NOT NULL,

        source_count BIGINT NOT NULL,
        valid_before_dedup_count BIGINT NOT NULL,
        rejected_count BIGINT NOT NULL,
        duplicate_count BIGINT NOT NULL,
        valid_after_dedup_count BIGINT NOT NULL,

        incoming_keys_found_in_silver BIGINT,
        reconciliation_status STRING NOT NULL,

        exploration_status STRING,
        expectation_status STRING,

        error_message STRING,

        started_at TIMESTAMP NOT NULL,
        completed_at TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (
        days(started_at)
    )
    TBLPROPERTIES (
        'format-version' = '2',
        'write.format.default' = 'parquet'
    )
    """
)

print(
    f"Silver audit table ready: "
    f"{SILVER_AUDIT_TABLE}"
)


# ============================================================
# VERIFY THAT ALL TARGET TABLES EXIST
# ============================================================

required_tables = [
    SILVER_TABLE,
    REJECTED_TABLE,
    SILVER_AUDIT_TABLE,
]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(
        table_name
    )
]

if missing_tables:
    raise RuntimeError(
        "The following Iceberg tables were not created: "
        f"{missing_tables}"
    )

print(
    "\nTarget-table creation validation: PASSED"
)


# ============================================================
# DISPLAY CREATED TABLES
# ============================================================

print("\nSilver namespace tables:")

spark.sql(
    f"SHOW TABLES IN {CATALOG}.silver"
).show(
    truncate=False
)

print("\nQuarantine namespace tables:")

spark.sql(
    f"SHOW TABLES IN {CATALOG}.quarantine"
).show(
    truncate=False
)

print("\nOperations namespace tables:")

spark.sql(
    f"SHOW TABLES IN {CATALOG}.operations"
).show(
    truncate=False
)


# ============================================================
# DISPLAY TABLE DESCRIPTIONS
# ============================================================

print("\nSilver table description:")

spark.sql(
    f"DESCRIBE TABLE {SILVER_TABLE}"
).show(
    100,
    truncate=False
)

print("\nRejected table description:")

spark.sql(
    f"DESCRIBE TABLE {REJECTED_TABLE}"
).show(
    100,
    truncate=False
)

print("\nSilver audit table description:")

spark.sql(
    f"DESCRIBE TABLE {SILVER_AUDIT_TABLE}"
).show(
    100,
    truncate=False
)

Namespace ready: datalake.silver
Namespace ready: datalake.quarantine
Namespace ready: datalake.operations
Silver table ready: datalake.silver.sales_order_lines
Rejected table ready: datalake.quarantine.sales_order_lines_rejected
Silver audit table ready: datalake.operations.silver_ingestion_audit

Target-table creation validation: PASSED

Silver namespace tables:
+---------+-----------------+-----------+
|namespace|tableName        |isTemporary|
+---------+-----------------+-----------+
|silver   |sales_order_lines|false      |
+---------+-----------------+-----------+


Quarantine namespace tables:
+----------+--------------------------+-----------+
|namespace |tableName                 |isTemporary|
+----------+--------------------------+-----------+
|quarantine|sales_order_lines_rejected|false      |
+----------+--------------------------+-----------+


Operations namespace tables:
+----------+----------------------+-----------+
|namespace |tableName             |isTemporary|
+----

In [12]:
# ============================================================
# ROW-LEVEL SILVER TRANSFORMATION AND VALIDATION
# ============================================================

def transform_and_validate_sales_order_lines(
    batch_df,
    *,
    stream_batch_id,
    pipeline_run_id
):
    """
    Transform one static foreachBatch DataFrame.

    Responsibilities
    ----------------
    1. Preserve raw Odoo relationship fields.
    2. Parse order_id and product_id JSON arrays.
    3. Apply deterministic Silver validation rules.
    4. Collect every failed rule and rejection reason.
    5. Label each record as VALID or REJECTED.

    This function performs no writes and no actions.
    It returns a lazy transformed DataFrame.
    """

    # --------------------------------------------------------
    # PRESERVE RAW ODOO RELATIONSHIP VALUES
    #
    # Examples:
    # order_id:
    # [14, "S00014"]
    #
    # product_id:
    # [3538, "[85123A] WHITE HANGING HEART..."]
    # --------------------------------------------------------

    transformed_df = (
        batch_df

        .withColumnRenamed(
            "order_id",
            "order_id_raw"
        )

        .withColumnRenamed(
            "product_id",
            "product_id_raw"
        )
    )


    # --------------------------------------------------------
    # PARSE ODOO RELATIONSHIP COLUMNS
    #
    # get_json_object returns the requested JSON element as
    # text. The first array element is converted to BIGINT.
    # --------------------------------------------------------

    transformed_df = (
        transformed_df

        .withColumn(
            "parsed_sales_order_id",
            F.get_json_object(
                F.col("order_id_raw"),
                "$[0]"
            ).cast(T.LongType())
        )

        .withColumn(
            "parsed_sales_order_name",
            F.get_json_object(
                F.col("order_id_raw"),
                "$[1]"
            )
        )

        .withColumn(
            "parsed_product_id",
            F.get_json_object(
                F.col("product_id_raw"),
                "$[0]"
            ).cast(T.LongType())
        )

        .withColumn(
            "parsed_product_display_name",
            F.get_json_object(
                F.col("product_id_raw"),
                "$[1]"
            )
        )
    )


    # --------------------------------------------------------
    # DEFINE ROW-LEVEL VALIDATION RULES
    #
    # Each tuple contains:
    #
    # 1. Stable rule name
    # 2. Human-readable rejection reason
    # 3. Condition that indicates failure
    # --------------------------------------------------------

    validation_rules = [
        (
            "ID_NOT_NULL",
            "id must not be null",
            F.col("id").isNull()
        ),

        (
            "ORDER_ID_NOT_NULL",
            "order_id must not be null",
            F.col("order_id_raw").isNull()
        ),

        (
            "ORDER_ID_PARSEABLE",
            "order_id must contain a valid numeric Odoo relation ID",
            (
                F.col("order_id_raw").isNotNull()
                & F.col(
                    "parsed_sales_order_id"
                ).isNull()
            )
        ),

        (
            "PRODUCT_ID_NOT_NULL",
            "product_id must not be null",
            F.col("product_id_raw").isNull()
        ),

        (
            "PRODUCT_ID_PARSEABLE",
            "product_id must contain a valid numeric Odoo relation ID",
            (
                F.col("product_id_raw").isNotNull()
                & F.col(
                    "parsed_product_id"
                ).isNull()
            )
        ),

        (
            "PRODUCT_UOM_QTY_NOT_NULL",
            "product_uom_qty must not be null",
            F.col("product_uom_qty").isNull()
        ),

        (
            "PRICE_UNIT_NOT_NULL",
            "price_unit must not be null",
            F.col("price_unit").isNull()
        ),

        (
            "DISCOUNT_NOT_NULL",
            "discount must not be null",
            F.col("discount").isNull()
        ),

        (
            "DISCOUNT_VALID_RANGE",
            "discount must be between 0 and 100",
            (
                F.col("discount").isNotNull()
                & (
                    (F.col("discount") < F.lit(0.0))
                    | (F.col("discount") > F.lit(100.0))
                )
            )
        ),

        (
            "PRICE_SUBTOTAL_NOT_NULL",
            "price_subtotal must not be null",
            F.col("price_subtotal").isNull()
        ),

        (
            "CREATE_DATE_NOT_NULL",
            "create_date must not be null or must contain a valid timestamp",
            F.col("create_date").isNull()
        ),

        (
            "WRITE_DATE_NOT_NULL",
            "write_date must not be null or must contain a valid timestamp",
            F.col("write_date").isNull()
        ),

        (
            "QTY_DELIVERED_NOT_NULL",
            "qty_delivered must not be null",
            F.col("qty_delivered").isNull()
        ),

        (
            "QTY_INVOICED_NOT_NULL",
            "qty_invoiced must not be null",
            F.col("qty_invoiced").isNull()
        ),
    ]


    # --------------------------------------------------------
    # CONSTRUCT FAILED-RULE AND REJECTION-REASON ARRAYS
    #
    # array_compact removes null entries generated by rules
    # that passed.
    # --------------------------------------------------------

    failed_rule_expressions = [
        F.when(
            failure_condition,
            F.lit(rule_name)
        )
        for (
            rule_name,
            rejection_reason,
            failure_condition
        ) in validation_rules
    ]

    rejection_reason_expressions = [
        F.when(
            failure_condition,
            F.lit(rejection_reason)
        )
        for (
            rule_name,
            rejection_reason,
            failure_condition
        ) in validation_rules
    ]


    transformed_df = (
        transformed_df

        .withColumn(
            "failed_rule_names",
            F.array_compact(
                F.array(
                    *failed_rule_expressions
                )
            )
        )

        .withColumn(
            "rejection_reasons",
            F.array_compact(
                F.array(
                    *rejection_reason_expressions
                )
            )
        )

        .withColumn(
            "record_status",
            F.when(
                F.size(
                    F.col("failed_rule_names")
                ) == 0,
                F.lit("VALID")
            ).otherwise(
                F.lit("REJECTED")
            )
        )
    )


    # --------------------------------------------------------
    # ADD STREAMING AND PROCESSING LINEAGE
    # --------------------------------------------------------

    transformed_df = (
        transformed_df

        .withColumn(
            "stream_batch_id",
            F.lit(
                int(stream_batch_id)
            ).cast(T.LongType())
        )

        .withColumn(
            "pipeline_run_id",
            F.lit(pipeline_run_id)
        )

        .withColumn(
            "processed_at",
            F.current_timestamp()
        )
    )


    return transformed_df

In [13]:
# ============================================================
# STATIC FUNCTION TEST
#
# This validates the transformation before it is used inside
# foreachBatch.
# ============================================================

TEST_PIPELINE_RUN_ID = (
    "static-validation-"
    + str(uuid.uuid4())
)

transformed_test_df = (
    transform_and_validate_sales_order_lines(
        bronze_static_df,
        stream_batch_id=-1,
        pipeline_run_id=TEST_PIPELINE_RUN_ID
    )
)


# ============================================================
# DISPLAY VALIDATION SUMMARY
# ============================================================

print("Static row-validation summary:")

transformed_test_df.groupBy(
    "record_status"
).count().orderBy(
    "record_status"
).show(
    truncate=False
)


# ============================================================
# DISPLAY PARSED ODOO RELATIONSHIPS
# ============================================================

print(
    "\nParsed Odoo relationship preview:"
)

transformed_test_df.select(
    "id",
    "order_id_raw",
    "parsed_sales_order_id",
    "parsed_sales_order_name",
    "product_id_raw",
    "parsed_product_id",
    "parsed_product_display_name",
    "record_status"
).filter(
    F.col("record_status") == "VALID"
).show(
    10,
    truncate=False
)


# ============================================================
# DISPLAY REJECTED TEST RECORDS
# ============================================================

print("\nRejected-record preview:")

transformed_test_df.select(
    "id",
    "order_id_raw",
    "product_id_raw",
    "product_uom_qty",
    "price_unit",
    "discount",
    "price_subtotal",
    "create_date",
    "write_date",
    "qty_delivered",
    "qty_invoiced",
    "failed_rule_names",
    "rejection_reasons",
    "batch_id",
    "_bronze_source_file"
).filter(
    F.col("record_status") == "REJECTED"
).show(
    50,
    truncate=False,
    vertical=True
)

Static row-validation summary:
+-------------+-----+
|record_status|count|
+-------------+-----+
|REJECTED     |6    |
|VALID        |10491|
+-------------+-----+


Parsed Odoo relationship preview:
+---+--------------+---------------------+-----------------------+------------------------------------------------------+-----------------+--------------------------------------------+-------------+
|id |order_id_raw  |parsed_sales_order_id|parsed_sales_order_name|product_id_raw                                        |parsed_product_id|parsed_product_display_name                 |record_status|
+---+--------------+---------------------+-----------------------+------------------------------------------------------+-----------------+--------------------------------------------+-------------+
|1  |[14, "S00014"]|14                   |S00014                 |[3538, "[85123A] WHITE HANGING HEART T-LIGHT HOLDER"] |3538             |[85123A] WHITE HANGING HEART T-LIGHT HOLDER |VALID        |
|2  |

In [14]:
# ============================================================
# BUILD VALID SILVER DATAFRAME
# ============================================================

def create_valid_silver_df(
    transformed_df
):
    """
    Select VALID records and convert them into the exact
    Silver table structure.

    This function performs no actions and no writes.
    """

    return (
        transformed_df

        .filter(
            F.col("record_status") == "VALID"
        )

        .select(
            F.col("id")
            .cast(T.LongType())
            .alias("sales_order_line_id"),

            F.col("parsed_sales_order_id")
            .cast(T.LongType())
            .alias("sales_order_id"),

            F.col("parsed_sales_order_name")
            .cast(T.StringType())
            .alias("sales_order_name"),

            F.col("parsed_product_id")
            .cast(T.LongType())
            .alias("product_id"),

            F.col("parsed_product_display_name")
            .cast(T.StringType())
            .alias("product_display_name"),

            F.col("product_uom_qty")
            .cast(T.DoubleType())
            .alias("product_uom_qty"),

            F.col("price_unit")
            .cast(T.DoubleType())
            .alias("price_unit"),

            F.col("discount")
            .cast(T.DoubleType())
            .alias("discount"),

            F.col("price_subtotal")
            .cast(T.DoubleType())
            .alias("price_subtotal"),

            F.col("create_date")
            .cast(T.TimestampType())
            .alias("create_date"),

            F.col("write_date")
            .cast(T.TimestampType())
            .alias("write_date"),

            F.col("qty_delivered")
            .cast(T.DoubleType())
            .alias("qty_delivered"),

            F.col("qty_invoiced")
            .cast(T.DoubleType())
            .alias("qty_invoiced"),

            F.col("batch_id")
            .cast(T.StringType())
            .alias("source_batch_id"),

            F.col("_bronze_source_file")
            .cast(T.StringType())
            .alias("source_file"),

            F.col("stream_batch_id")
            .cast(T.LongType())
            .alias("stream_batch_id"),

            F.col("pipeline_run_id")
            .cast(T.StringType())
            .alias("pipeline_run_id"),

            # These values are used when inserting a new row.
            # During MERGE, silver_created_at will be preserved
            # for existing rows.
            F.col("processed_at")
            .cast(T.TimestampType())
            .alias("silver_created_at"),

            F.col("processed_at")
            .cast(T.TimestampType())
            .alias("silver_updated_at"),
        )
    )


# ============================================================
# BUILD REJECTED-RECORD DATAFRAME
# ============================================================

def create_rejected_records_df(
    transformed_df
):
    """
    Select REJECTED records and convert them into the exact
    quarantine table structure.

    This function performs no actions and no writes.
    """

    return (
        transformed_df

        .filter(
            F.col("record_status") == "REJECTED"
        )

        .select(
            F.col("id")
            .cast(T.LongType())
            .alias("id"),

            F.col("order_id_raw")
            .cast(T.StringType())
            .alias("order_id_raw"),

            F.col("product_id_raw")
            .cast(T.StringType())
            .alias("product_id_raw"),

            F.col("product_uom_qty")
            .cast(T.DoubleType())
            .alias("product_uom_qty"),

            F.col("price_unit")
            .cast(T.DoubleType())
            .alias("price_unit"),

            F.col("discount")
            .cast(T.DoubleType())
            .alias("discount"),

            F.col("price_subtotal")
            .cast(T.DoubleType())
            .alias("price_subtotal"),

            F.col("create_date")
            .cast(T.TimestampType())
            .alias("create_date"),

            F.col("write_date")
            .cast(T.TimestampType())
            .alias("write_date"),

            F.col("qty_delivered")
            .cast(T.DoubleType())
            .alias("qty_delivered"),

            F.col("qty_invoiced")
            .cast(T.DoubleType())
            .alias("qty_invoiced"),

            F.col("parsed_sales_order_id")
            .cast(T.LongType())
            .alias("parsed_sales_order_id"),

            F.col("parsed_sales_order_name")
            .cast(T.StringType())
            .alias("parsed_sales_order_name"),

            F.col("parsed_product_id")
            .cast(T.LongType())
            .alias("parsed_product_id"),

            F.col("parsed_product_display_name")
            .cast(T.StringType())
            .alias("parsed_product_display_name"),

            F.col("rejection_reasons")
            .alias("rejection_reasons"),

            F.col("failed_rule_names")
            .alias("failed_rule_names"),

            F.col("batch_id")
            .cast(T.StringType())
            .alias("source_batch_id"),

            F.col("_bronze_source_file")
            .cast(T.StringType())
            .alias("source_file"),

            F.col("stream_batch_id")
            .cast(T.LongType())
            .alias("stream_batch_id"),

            F.col("pipeline_run_id")
            .cast(T.StringType())
            .alias("pipeline_run_id"),

            F.col("processed_at")
            .cast(T.TimestampType())
            .alias("rejected_at"),
        )
    )

In [15]:
# ============================================================
# CREATE STATIC TEST DATAFRAMES
# ============================================================

valid_silver_test_df = (
    create_valid_silver_df(
        transformed_test_df
    )
)

rejected_records_test_df = (
    create_rejected_records_df(
        transformed_test_df
    )
)


# ============================================================
# MATERIALIZE TEST COUNTS
# ============================================================

transformed_test_count = (
    transformed_test_df.count()
)

valid_test_count = (
    valid_silver_test_df.count()
)

rejected_test_count = (
    rejected_records_test_df.count()
)


print("Static split validation")
print("-" * 70)

print(
    "Transformed source count:",
    transformed_test_count
)

print(
    "Valid Silver count:",
    valid_test_count
)

print(
    "Rejected-record count:",
    rejected_test_count
)


# ============================================================
# RECONCILE STATIC SPLIT
# ============================================================

if (
    transformed_test_count
    != valid_test_count
    + rejected_test_count
):
    raise ValueError(
        "Static split reconciliation failed: "
        f"{transformed_test_count} != "
        f"{valid_test_count} + "
        f"{rejected_test_count}"
    )

print(
    "Static split reconciliation: PASSED"
)


# ============================================================
# VALIDATE DATAFRAME COLUMN ORDER
# ============================================================

expected_silver_columns = [
    field.name
    for field in silver_schema.fields
]

expected_rejected_columns = [
    field.name
    for field in rejected_schema.fields
]


if (
    valid_silver_test_df.columns
    != expected_silver_columns
):
    raise ValueError(
        "Valid Silver DataFrame columns do not match "
        "the Silver table schema.\n"
        f"Expected: {expected_silver_columns}\n"
        f"Actual:   {valid_silver_test_df.columns}"
    )


if (
    rejected_records_test_df.columns
    != expected_rejected_columns
):
    raise ValueError(
        "Rejected DataFrame columns do not match "
        "the rejected table schema.\n"
        f"Expected: {expected_rejected_columns}\n"
        f"Actual:   {rejected_records_test_df.columns}"
    )


print(
    "Silver column validation: PASSED"
)

print(
    "Rejected column validation: PASSED"
)


# ============================================================
# DISPLAY VALID SILVER PREVIEW
# ============================================================

print("\nValid Silver preview:")

valid_silver_test_df.select(
    "sales_order_line_id",
    "sales_order_id",
    "sales_order_name",
    "product_id",
    "product_display_name",
    "product_uom_qty",
    "price_unit",
    "discount",
    "price_subtotal",
    "write_date",
    "source_batch_id"
).show(
    10,
    truncate=False
)


# ============================================================
# DISPLAY REJECTED RECORD PREVIEW
# ============================================================

print("\nRejected-record preview:")

rejected_records_test_df.select(
    "id",
    "order_id_raw",
    "product_id_raw",
    "failed_rule_names",
    "rejection_reasons",
    "source_batch_id",
    "source_file"
).show(
    50,
    truncate=False,
    vertical=True
)

Static split validation
----------------------------------------------------------------------
Transformed source count: 10497
Valid Silver count: 10491
Rejected-record count: 6
Static split reconciliation: PASSED
Silver column validation: PASSED
Rejected column validation: PASSED

Valid Silver preview:
+-------------------+--------------+----------------+----------+--------------------------------------------+---------------+----------+--------+--------------+-------------------+---------------+
|sales_order_line_id|sales_order_id|sales_order_name|product_id|product_display_name                        |product_uom_qty|price_unit|discount|price_subtotal|write_date         |source_batch_id|
+-------------------+--------------+----------------+----------+--------------------------------------------+---------------+----------+--------+--------------+-------------------+---------------+
|1                  |14            |S00014          |3538      |[85123A] WHITE HANGING HEART T-LIGHT HOL

In [16]:
# ============================================================
# DEDUPLICATE VALID SILVER RECORDS
# ============================================================

def deduplicate_valid_records(
    valid_df
):
    """
    Keep one record per sales_order_line_id.

    Selection priority
    ------------------
    1. Latest write_date
    2. Latest source_batch_id for deterministic tie-breaking
    3. Source-file path for deterministic tie-breaking

    Returns
    -------
    deduplicated_df:
        One latest record per sales_order_line_id.

    duplicate_records_df:
        Records removed as duplicates.
    """

    deduplication_window = (
        Window
        .partitionBy(
            "sales_order_line_id"
        )
        .orderBy(
            F.col("write_date").desc(),
            F.col("source_batch_id").desc(),
            F.col("source_file").desc()
        )
    )

    ranked_df = (
        valid_df
        .withColumn(
            "_deduplication_rank",
            F.row_number().over(
                deduplication_window
            )
        )
    )

    deduplicated_df = (
        ranked_df
        .filter(
            F.col("_deduplication_rank") == 1
        )
        .drop(
            "_deduplication_rank"
        )
    )

    duplicate_records_df = (
        ranked_df
        .filter(
            F.col("_deduplication_rank") > 1
        )
        .drop(
            "_deduplication_rank"
        )
    )

    return (
        deduplicated_df,
        duplicate_records_df
    )

In [17]:
# ============================================================
# APPLY DEDUPLICATION TO STATIC TEST DATA
# ============================================================

(
    deduplicated_test_df,
    duplicate_test_df
) = deduplicate_valid_records(
    valid_silver_test_df
)


# Cache only for this static validation because several
# actions are executed below.
deduplicated_test_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

duplicate_test_df.persist(
    StorageLevel.MEMORY_AND_DISK
)


try:
    # --------------------------------------------------------
    # MATERIALIZE COUNTS
    # --------------------------------------------------------

    valid_before_dedup_count = (
        valid_silver_test_df.count()
    )

    valid_after_dedup_count = (
        deduplicated_test_df.count()
    )

    duplicate_count = (
        duplicate_test_df.count()
    )


    print("Static deduplication summary")
    print("-" * 70)

    print(
        "Valid before deduplication:",
        valid_before_dedup_count
    )

    print(
        "Duplicate records removed:",
        duplicate_count
    )

    print(
        "Valid after deduplication:",
        valid_after_dedup_count
    )


    # --------------------------------------------------------
    # RECONCILE DEDUPLICATION
    # --------------------------------------------------------

    if (
        valid_before_dedup_count
        != valid_after_dedup_count
        + duplicate_count
    ):
        raise ValueError(
            "Deduplication reconciliation failed: "
            f"{valid_before_dedup_count} != "
            f"{valid_after_dedup_count} + "
            f"{duplicate_count}"
        )

    print(
        "Deduplication reconciliation: PASSED"
    )


    # --------------------------------------------------------
    # VERIFY UNIQUE PRIMARY KEYS
    # --------------------------------------------------------

    remaining_duplicate_keys = (
        deduplicated_test_df
        .groupBy(
            "sales_order_line_id"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    print(
        "Duplicate IDs remaining:",
        remaining_duplicate_keys
    )

    if remaining_duplicate_keys > 0:
        raise ValueError(
            "Duplicate sales_order_line_id values remain "
            "after deduplication."
        )

    print(
        "Primary-key uniqueness validation: PASSED"
    )


    # --------------------------------------------------------
    # DISPLAY DUPLICATE DISTRIBUTION
    # --------------------------------------------------------

    print(
        "\nDuplicate records by source batch:"
    )

    duplicate_test_df.groupBy(
        "source_batch_id"
    ).count().orderBy(
        F.col("count").desc(),
        F.col("source_batch_id")
    ).show(
        truncate=False
    )


    # --------------------------------------------------------
    # SHOW SAMPLE DUPLICATE GROUPS
    # --------------------------------------------------------

    duplicate_id_sample = [
        row["sales_order_line_id"]
        for row in (
            valid_silver_test_df
            .groupBy(
                "sales_order_line_id"
            )
            .count()
            .filter(
                F.col("count") > 1
            )
            .orderBy(
                "sales_order_line_id"
            )
            .limit(5)
            .collect()
        )
    ]

    print(
        "\nSample duplicate IDs:",
        duplicate_id_sample
    )

    if duplicate_id_sample:
        print(
            "\nRecords before deduplication:"
        )

        valid_silver_test_df.filter(
            F.col(
                "sales_order_line_id"
            ).isin(
                duplicate_id_sample
            )
        ).select(
            "sales_order_line_id",
            "write_date",
            "source_batch_id",
            "source_file"
        ).orderBy(
            "sales_order_line_id",
            F.col("write_date").desc(),
            F.col("source_batch_id").desc()
        ).show(
            50,
            truncate=False
        )

        print(
            "\nWinning records after deduplication:"
        )

        deduplicated_test_df.filter(
            F.col(
                "sales_order_line_id"
            ).isin(
                duplicate_id_sample
            )
        ).select(
            "sales_order_line_id",
            "write_date",
            "source_batch_id",
            "source_file"
        ).orderBy(
            "sales_order_line_id"
        ).show(
            50,
            truncate=False
        )

finally:
    deduplicated_test_df.unpersist()
    duplicate_test_df.unpersist()

Static deduplication summary
----------------------------------------------------------------------
Valid before deduplication: 10491
Duplicate records removed: 6994
Valid after deduplication: 3497
Deduplication reconciliation: PASSED
Duplicate IDs remaining: 0
Primary-key uniqueness validation: PASSED

Duplicate records by source batch:
+------------------+-----+
|source_batch_id   |count|
+------------------+-----+
|api-00002         |3497 |
|api-00002-rerun-01|3497 |
+------------------+-----+


Sample duplicate IDs: [1, 2, 3, 4, 5]

Records before deduplication:
+-------------------+-------------------+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|sales_order_line_id|write_date         |source_batch_id   |source_file                                                                                                     

In [28]:
# ============================================================
# ICEBERG MERGE USING A MATERIALIZED STAGING TABLE
# ============================================================

def merge_sales_order_lines_to_silver(
    spark_session,
    deduplicated_df,
    *,
    silver_table,
    stream_batch_id
):
    """
    Materialize the deduplicated micro-batch into a temporary
    Iceberg staging table and MERGE that table into Silver.

    The staging table removes non-deterministic expressions,
    such as current_timestamp(), from the MERGE source plan.

    Merge key
    ---------
    sales_order_line_id

    Update rule
    -----------
    Update the existing Silver record only when the incoming
    write_date is newer.

    Equal or older incoming versions do not update Silver.
    """

    # --------------------------------------------------------
    # CREATE A UNIQUE STAGING TABLE NAME
    # --------------------------------------------------------

    staging_suffix = (
        f"{PIPELINE_RUN_ID}_{stream_batch_id}"
        .replace("-", "_")
    )

    staging_table = (
        f"{CATALOG}.operations."
        f"sales_order_lines_staging_{staging_suffix}"
    )

    print(
        "Silver staging table:",
        staging_table
    )


    # --------------------------------------------------------
    # VALIDATE REQUIRED MERGE COLUMNS
    # --------------------------------------------------------

    required_merge_columns = {
        "sales_order_line_id",
        "write_date",
        "silver_created_at",
        "silver_updated_at",
    }

    missing_merge_columns = (
        required_merge_columns
        - set(deduplicated_df.columns)
    )

    if missing_merge_columns:
        raise ValueError(
            "Required MERGE columns are missing: "
            f"{sorted(missing_merge_columns)}"
        )


    # --------------------------------------------------------
    # VALIDATE NULL PRIMARY KEYS
    # --------------------------------------------------------

    null_key_count = (
        deduplicated_df
        .filter(
            F.col(
                "sales_order_line_id"
            ).isNull()
        )
        .count()
    )

    if null_key_count > 0:
        raise ValueError(
            "The MERGE source contains null "
            "sales_order_line_id values."
        )


    # --------------------------------------------------------
    # VALIDATE PRIMARY-KEY UNIQUENESS
    # --------------------------------------------------------

    duplicate_key_count = (
        deduplicated_df
        .groupBy(
            "sales_order_line_id"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    if duplicate_key_count > 0:
        raise ValueError(
            "The MERGE source still contains duplicate "
            "sales_order_line_id values."
        )


    # --------------------------------------------------------
    # REMOVE A LEFTOVER STAGING TABLE
    #
    # This supports retrying a previously failed micro-batch.
    # --------------------------------------------------------

    spark_session.sql(
        f"""
        DROP TABLE IF EXISTS {staging_table}
        """
    )


    try:
        # ----------------------------------------------------
        # MATERIALIZE THE SOURCE INTO ICEBERG
        #
        # create() executes the complete source plan and stores
        # fixed values for current_timestamp() and all derived
        # columns.
        # ----------------------------------------------------

        (
            deduplicated_df
            .writeTo(staging_table)
            .using("iceberg")
            .create()
        )

        spark_session.catalog.refreshTable(
            staging_table
        )

        staging_count = (
            spark_session
            .table(staging_table)
            .count()
        )

        expected_staging_count = (
            deduplicated_df.count()
        )

        print(
            "Expected staging count:",
            expected_staging_count
        )

        print(
            "Written staging count:",
            staging_count
        )

        if staging_count != expected_staging_count:
            raise ValueError(
                "Staging-table reconciliation failed: "
                f"expected {expected_staging_count}, "
                f"found {staging_count}"
            )


        # ----------------------------------------------------
        # EXECUTE MERGE FROM THE MATERIALIZED TABLE
        # ----------------------------------------------------

        spark_session.sql(
            f"""
            MERGE INTO {silver_table} AS target

            USING {staging_table} AS source

            ON (
                target.sales_order_line_id
                =
                source.sales_order_line_id
            )

            WHEN MATCHED
                AND source.write_date > target.write_date
            THEN UPDATE SET

                target.sales_order_id =
                    source.sales_order_id,

                target.sales_order_name =
                    source.sales_order_name,

                target.product_id =
                    source.product_id,

                target.product_display_name =
                    source.product_display_name,

                target.product_uom_qty =
                    source.product_uom_qty,

                target.price_unit =
                    source.price_unit,

                target.discount =
                    source.discount,

                target.price_subtotal =
                    source.price_subtotal,

                target.create_date =
                    source.create_date,

                target.write_date =
                    source.write_date,

                target.qty_delivered =
                    source.qty_delivered,

                target.qty_invoiced =
                    source.qty_invoiced,

                target.source_batch_id =
                    source.source_batch_id,

                target.source_file =
                    source.source_file,

                target.stream_batch_id =
                    source.stream_batch_id,

                target.pipeline_run_id =
                    source.pipeline_run_id,

                target.silver_updated_at =
                    source.silver_updated_at

            WHEN NOT MATCHED
            THEN INSERT
            (
                sales_order_line_id,
                sales_order_id,
                sales_order_name,
                product_id,
                product_display_name,
                product_uom_qty,
                price_unit,
                discount,
                price_subtotal,
                create_date,
                write_date,
                qty_delivered,
                qty_invoiced,
                source_batch_id,
                source_file,
                stream_batch_id,
                pipeline_run_id,
                silver_created_at,
                silver_updated_at
            )
            VALUES
            (
                source.sales_order_line_id,
                source.sales_order_id,
                source.sales_order_name,
                source.product_id,
                source.product_display_name,
                source.product_uom_qty,
                source.price_unit,
                source.discount,
                source.price_subtotal,
                source.create_date,
                source.write_date,
                source.qty_delivered,
                source.qty_invoiced,
                source.source_batch_id,
                source.source_file,
                source.stream_batch_id,
                source.pipeline_run_id,
                source.silver_created_at,
                source.silver_updated_at
            )
            """
        )

        spark_session.catalog.refreshTable(
            silver_table
        )

        print(
            "Silver MERGE completed from staging table."
        )

    finally:
        # ----------------------------------------------------
        # ALWAYS REMOVE THE TEMPORARY STAGING TABLE
        # ----------------------------------------------------

        try:
            spark_session.sql(
                f"""
                DROP TABLE IF EXISTS {staging_table}
                """
            )

            print(
                "Staging table removed:",
                staging_table
            )

        except Exception as staging_cleanup_error:
            print(
                "Warning: staging table cleanup failed:",
                staging_cleanup_error
            )

    return staging_table


# ============================================================
# VERIFY INCOMING KEYS AFTER MERGE
# ============================================================

def count_incoming_keys_found_in_silver(
    spark_session,
    incoming_df,
    *,
    silver_table
):
    """
    Count distinct incoming IDs that exist in Silver after
    MERGE.
    """

    incoming_keys_df = (
        incoming_df
        .select(
            "sales_order_line_id"
        )
        .dropDuplicates()
    )

    silver_keys_df = (
        spark_session
        .table(silver_table)
        .select(
            "sales_order_line_id"
        )
        .dropDuplicates()
    )

    return (
        incoming_keys_df
        .join(
            silver_keys_df,
            on="sales_order_line_id",
            how="left_semi"
        )
        .count()
    )


# ============================================================
# VERIFY INCOMING BUSINESS VERSIONS AFTER MERGE
# ============================================================

def count_incoming_versions_satisfied_in_silver(
    spark_session,
    incoming_df,
    *,
    silver_table
):
    """
    Count incoming IDs for which Silver contains the same or
    a newer write_date.
    """

    incoming_versions_df = (
        incoming_df
        .select(
            "sales_order_line_id",
            F.col("write_date").alias(
                "incoming_write_date"
            )
        )
        .dropDuplicates(
            ["sales_order_line_id"]
        )
    )

    silver_versions_df = (
        spark_session
        .table(silver_table)
        .select(
            "sales_order_line_id",
            F.col("write_date").alias(
                "silver_write_date"
            )
        )
    )

    return (
        incoming_versions_df
        .join(
            silver_versions_df,
            on="sales_order_line_id",
            how="inner"
        )
        .filter(
            F.col("silver_write_date")
            >= F.col("incoming_write_date")
        )
        .count()
    )


# ============================================================
# FUNCTION VALIDATION
# ============================================================

print(
    "Staging MERGE function:",
    merge_sales_order_lines_to_silver.__name__
)

print(
    "Key verification function:",
    count_incoming_keys_found_in_silver.__name__
)

print(
    "Version verification function:",
    count_incoming_versions_satisfied_in_silver.__name__
)

print(
    "\nMaterialized Iceberg MERGE utilities "
    "defined successfully."
)

Staging MERGE function: merge_sales_order_lines_to_silver
Key verification function: count_incoming_keys_found_in_silver
Version verification function: count_incoming_versions_satisfied_in_silver

Materialized Iceberg MERGE utilities defined successfully.


In [19]:
# ============================================================
# WRITE REJECTED RECORDS TO ICEBERG
# ============================================================

def write_rejected_records(
    rejected_df,
    *,
    rejected_table,
    rejected_count=None
):
    """
    Append rejected Silver records to the Iceberg quarantine
    table.

    Parameters
    ----------
    rejected_df:
        Static DataFrame produced inside foreachBatch.

    rejected_table:
        Fully qualified Iceberg table name.

    rejected_count:
        Optional known count. When supplied, the function
        avoids executing an additional count action.

    Returns
    -------
    Number of rejected records written.
    """

    if rejected_count is None:
        rejected_count = rejected_df.count()

    if rejected_count == 0:
        print(
            "No rejected records found. "
            "Rejected-record write skipped."
        )

        return 0

    # Confirm that the DataFrame columns exactly match
    # the rejected Iceberg table schema.
    expected_columns = [
        field.name
        for field in rejected_schema.fields
    ]

    if rejected_df.columns != expected_columns:
        raise ValueError(
            "Rejected DataFrame columns do not match "
            "the rejected-table schema.\n"
            f"Expected: {expected_columns}\n"
            f"Actual:   {rejected_df.columns}"
        )

    (
        rejected_df
        .writeTo(rejected_table)
        .append()
    )

    print(
        f"Rejected records written: {rejected_count}"
    )

    return rejected_count


# ============================================================
# CREATE A SILVER AUDIT ROW
# ============================================================

def create_silver_audit_row(
    *,
    pipeline_run_id,
    stream_batch_id,
    status,
    source_count,
    valid_before_dedup_count,
    rejected_count,
    duplicate_count,
    valid_after_dedup_count,
    incoming_keys_found_in_silver=None,
    reconciliation_status,
    exploration_status=None,
    expectation_status=None,
    error_message=None,
    started_at,
    completed_at=None
):
    """
    Create one dictionary matching silver_audit_schema.
    """

    return {
        "audit_id": str(uuid.uuid4()),

        "pipeline_run_id": str(
            pipeline_run_id
        ),

        "stream_batch_id": int(
            stream_batch_id
        ),

        "dataset_name": DATASET_NAME,

        "processing_stage": (
            "BRONZE_TO_SILVER"
        ),

        "status": status,

        "source_count": int(
            source_count
        ),

        "valid_before_dedup_count": int(
            valid_before_dedup_count
        ),

        "rejected_count": int(
            rejected_count
        ),

        "duplicate_count": int(
            duplicate_count
        ),

        "valid_after_dedup_count": int(
            valid_after_dedup_count
        ),

        "incoming_keys_found_in_silver": (
            int(incoming_keys_found_in_silver)
            if incoming_keys_found_in_silver is not None
            else None
        ),

        "reconciliation_status": (
            reconciliation_status
        ),

        "exploration_status": (
            exploration_status
        ),

        "expectation_status": (
            expectation_status
        ),

        "error_message": (
            error_message
        ),

        "started_at": started_at,

        "completed_at": completed_at,
    }


# ============================================================
# WRITE SILVER MICRO-BATCH AUDIT
# ============================================================

def write_silver_audit(
    spark_session,
    audit_row,
    *,
    audit_table
):
    """
    Write one Bronze-to-Silver micro-batch audit record to the
    Iceberg audit table.
    """

    audit_df = (
        spark_session
        .createDataFrame(
            [audit_row],
            schema=silver_audit_schema
        )
    )

    (
        audit_df
        .writeTo(audit_table)
        .append()
    )

    print(
        "Silver audit record written:",
        audit_row["audit_id"]
    )

    return audit_row["audit_id"]


# ============================================================
# RECONCILIATION HELPER
# ============================================================

def evaluate_micro_batch_reconciliation(
    *,
    source_count,
    valid_before_dedup_count,
    rejected_count,
    duplicate_count,
    valid_after_dedup_count,
    incoming_keys_found_in_silver
):
    """
    Evaluate the three Bronze-to-Silver reconciliation checks.

    Checks
    ------
    1. Source split:
       source_count =
           valid_before_dedup_count + rejected_count

    2. Deduplication:
       valid_before_dedup_count =
           valid_after_dedup_count + duplicate_count

    3. Silver key verification:
       valid_after_dedup_count =
           incoming_keys_found_in_silver
    """

    source_split_passed = (
        source_count
        ==
        valid_before_dedup_count
        + rejected_count
    )

    deduplication_passed = (
        valid_before_dedup_count
        ==
        valid_after_dedup_count
        + duplicate_count
    )

    silver_merge_passed = (
        valid_after_dedup_count
        ==
        incoming_keys_found_in_silver
    )

    reconciliation_status = (
        "RECONCILED"
        if (
            source_split_passed
            and deduplication_passed
            and silver_merge_passed
        )
        else "FAILED"
    )

    reconciliation_details = {
        "source_split_passed": (
            source_split_passed
        ),

        "deduplication_passed": (
            deduplication_passed
        ),

        "silver_merge_passed": (
            silver_merge_passed
        ),

        "reconciliation_status": (
            reconciliation_status
        ),
    }

    return reconciliation_details


# ============================================================
# FUNCTION-DEFINITION VALIDATION
# ============================================================

print(
    "Rejected writer:",
    write_rejected_records.__name__
)

print(
    "Audit-row builder:",
    create_silver_audit_row.__name__
)

print(
    "Audit writer:",
    write_silver_audit.__name__
)

print(
    "Reconciliation helper:",
    evaluate_micro_batch_reconciliation.__name__
)

print(
    "\nRejected-record and audit utilities "
    "defined successfully."
)

Rejected writer: write_rejected_records
Audit-row builder: create_silver_audit_row
Audit writer: write_silver_audit
Reconciliation helper: evaluate_micro_batch_reconciliation

Rejected-record and audit utilities defined successfully.


In [20]:
# ============================================================
# PIPELINE RUN ID
#
# One run ID identifies this execution of the streaming query.
# All micro-batches processed by this query share this run ID.
# ============================================================

import uuid

PIPELINE_RUN_ID = (
    "sales-order-lines-"
    + str(uuid.uuid4())
)

print(
    "Pipeline run ID:",
    PIPELINE_RUN_ID
)


# ============================================================
# EXPLORATION CONFIGURATION
# ============================================================

EXPLORATION_COLUMNS = [
    "id",
    "order_id",
    "product_id",
    "product_uom_qty",
    "price_unit",
    "discount",
    "price_subtotal",
    "create_date",
    "write_date",
    "qty_delivered",
    "qty_invoiced",
]

EXPLORATION_REQUIRED_COLUMNS = [
    "id",
    "order_id",
    "product_id",
    "product_uom_qty",
    "price_unit",
    "discount",
    "price_subtotal",
    "create_date",
    "write_date",
    "qty_delivered",
    "qty_invoiced",
]


# ============================================================
# MICRO-BATCH PROCESSING FUNCTION
# ============================================================

def process_sales_order_lines_batch(
    batch_df,
    stream_batch_id
):
    """
    Process one static Bronze micro-batch supplied by
    Structured Streaming foreachBatch.

    Processing order
    ----------------
    1. Cache and materialize the Bronze micro-batch.
    2. Add source-file and Bronze batch metadata.
    3. Run data exploration.
    4. Run Great Expectations monitoring.
    5. Parse Odoo relationship columns.
    6. Apply row-level Silver validation.
    7. Split valid and rejected records.
    8. Write rejected records to Iceberg quarantine.
    9. Deduplicate valid records.
    10. MERGE valid records into Silver.
    11. Verify incoming keys and versions.
    12. Perform reconciliation.
    13. Write the micro-batch audit record.
    14. Release callback-owned caches.
    """

    batch_started_at = datetime.utcnow()

    # Convert Spark's batch ID into a normal Python integer.
    stream_batch_id = int(stream_batch_id)

    batch_run_id = (
        f"{PIPELINE_RUN_ID}-batch-"
        f"{stream_batch_id}"
    )

    print("\n" + "=" * 80)
    print(
        "Processing stream batch:",
        stream_batch_id
    )
    print(
        "Batch run ID:",
        batch_run_id
    )
    print("=" * 80)

    # --------------------------------------------------------
    # INITIAL AUDIT VALUES
    #
    # These defaults allow a failure audit to be written even
    # when processing fails before all counts are calculated.
    # --------------------------------------------------------

    source_count = 0
    valid_before_dedup_count = 0
    rejected_count = 0
    duplicate_count = 0
    valid_after_dedup_count = 0
    incoming_keys_found_in_silver = None
    incoming_versions_satisfied = None

    exploration_status = "NOT_STARTED"
    expectation_status = "NOT_STARTED"
    reconciliation_status = "NOT_EVALUATED"

    transformed_df = None
    valid_df = None
    rejected_df = None
    deduplicated_df = None
    duplicate_df = None

    # --------------------------------------------------------
    # CACHE THE CALLBACK-OWNED MICRO-BATCH
    # --------------------------------------------------------

    batch_df.persist(
        StorageLevel.MEMORY_AND_DISK
    )

    try:
        # Materialize the micro-batch once.
        source_count = batch_df.count()

        print(
            "Source records:",
            source_count
        )

        # An empty micro-batch requires no data processing.
        if source_count == 0:
            empty_audit_row = create_silver_audit_row(
                pipeline_run_id=PIPELINE_RUN_ID,
                stream_batch_id=stream_batch_id,
                status="SKIPPED_EMPTY_BATCH",
                source_count=0,
                valid_before_dedup_count=0,
                rejected_count=0,
                duplicate_count=0,
                valid_after_dedup_count=0,
                incoming_keys_found_in_silver=0,
                reconciliation_status="RECONCILED",
                exploration_status="SKIPPED",
                expectation_status="SKIPPED",
                error_message=None,
                started_at=batch_started_at,
                completed_at=datetime.utcnow(),
            )

            write_silver_audit(
                batch_df.sparkSession,
                empty_audit_row,
                audit_table=SILVER_AUDIT_TABLE
            )

            print(
                "Empty micro-batch skipped."
            )

            return

        batch_spark = batch_df.sparkSession

        # ----------------------------------------------------
        # ADD SOURCE LINEAGE AND BRONZE PATH METADATA
        #
        # recursiveFileLookup disables automatic partition
        # discovery, so values are extracted from file paths.
        # ----------------------------------------------------

        enriched_batch_df = (
            batch_df

            .withColumn(
                "_bronze_source_file",
                F.input_file_name()
            )

            .withColumn(
                "batch_id",
                F.regexp_extract(
                    F.col(
                        "_bronze_source_file"
                    ),
                    r"batch_id=([^/]+)",
                    1
                )
            )

            .withColumn(
                "write_date_partition",
                F.to_date(
                    F.regexp_extract(
                        F.col(
                            "_bronze_source_file"
                        ),
                        r"write_date_partition=([^/]+)",
                        1
                    ),
                    "yyyy-MM-dd"
                )
            )
        )

        # ----------------------------------------------------
        # DATA EXPLORATION
        #
        # The profiler returns a lazy measures DataFrame.
        # The writer persists those measures into:
        # datalake.operations.explorations
        # ----------------------------------------------------

        print("\nRunning data exploration...")

        try:
            exploration_results_df = explore_data(
                batch_spark,
                enriched_batch_df,
                dataset_name=DATASET_NAME,
                run_id=batch_run_id,
                batch_id=str(stream_batch_id),
                profile_scope="BATCH",
                processing_stage="BRONZE",
                profile_level="FULL",
                required_columns=(
                    EXPLORATION_REQUIRED_COLUMNS
                ),
                columns=EXPLORATION_COLUMNS,
                approximate_distinct=True,
                relative_sd=0.05,
            )

            write_exploration_to_iceberg(
                batch_spark,
                exploration_results_df,
                catalog_name=CATALOG,
                namespace_name="operations",
                table_name="explorations",
            )

            exploration_status = "COMPLETED"

            print(
                "Data exploration completed."
            )

        except Exception as exploration_error:
            # Profiling failure is recorded but does not prevent
            # valid business records from reaching Silver.
            exploration_status = (
                "FAILED: "
                + type(
                    exploration_error
                ).__name__
            )

            print(
                "Data exploration failed:",
                exploration_error
            )

        # ----------------------------------------------------
        # GREAT EXPECTATIONS PROFILING
        #
        # Failed expectations are quality findings. The mentor
        # profiler writes PASSED/FAILED and per-rule results to
        # the expectation_profiles Iceberg table.
        # ----------------------------------------------------

        print(
            "\nRunning Great Expectations..."
        )

        try:
            profile_expectations(
                enriched_batch_df,
                gx_expectations,
                dataset_name=DATASET_NAME,
                run_id=batch_run_id,
                audit_table=EXPECTATION_TABLE,
            )

            expectation_status = "COMPLETED"

            print(
                "Great Expectations profiling completed."
            )

        except Exception as expectation_error:
            # GX configuration, execution, or audit-write errors
            # are recorded but do not stop transformation.
            expectation_status = (
                "FAILED: "
                + type(
                    expectation_error
                ).__name__
            )

            print(
                "Great Expectations profiling failed:",
                expectation_error
            )

        # ----------------------------------------------------
        # TRANSFORM AND VALIDATE
        # ----------------------------------------------------

        print(
            "\nTransforming and validating records..."
        )

        transformed_df = (
            transform_and_validate_sales_order_lines(
                enriched_batch_df,
                stream_batch_id=stream_batch_id,
                pipeline_run_id=PIPELINE_RUN_ID,
            )
        )

        transformed_df.persist(
            StorageLevel.MEMORY_AND_DISK
        )

        # Materialize the transformed DataFrame before creating
        # multiple downstream branches.
        transformed_count = (
            transformed_df.count()
        )

        if transformed_count != source_count:
            raise ValueError(
                "Transformation changed the record count: "
                f"source={source_count}, "
                f"transformed={transformed_count}"
            )

        # ----------------------------------------------------
        # SPLIT VALID AND REJECTED RECORDS
        # ----------------------------------------------------

        valid_df = create_valid_silver_df(
            transformed_df
        )

        rejected_df = create_rejected_records_df(
            transformed_df
        )

        valid_df.persist(
            StorageLevel.MEMORY_AND_DISK
        )

        rejected_df.persist(
            StorageLevel.MEMORY_AND_DISK
        )

        valid_before_dedup_count = (
            valid_df.count()
        )

        rejected_count = (
            rejected_df.count()
        )

        print(
            "Valid before deduplication:",
            valid_before_dedup_count
        )

        print(
            "Rejected records:",
            rejected_count
        )

        # ----------------------------------------------------
        # VERIFY SOURCE SPLIT BEFORE WRITING
        # ----------------------------------------------------

        if (
            source_count
            != valid_before_dedup_count
            + rejected_count
        ):
            raise ValueError(
                "Source split reconciliation failed: "
                f"{source_count} != "
                f"{valid_before_dedup_count} + "
                f"{rejected_count}"
            )

        # ----------------------------------------------------
        # WRITE SILVER-REJECTED RECORDS
        # ----------------------------------------------------

        write_rejected_records(
            rejected_df,
            rejected_table=REJECTED_TABLE,
            rejected_count=rejected_count,
        )

        # ----------------------------------------------------
        # DEDUPLICATE VALID RECORDS
        #
        # One record is retained per sales_order_line_id.
        # Latest write_date wins.
        # ----------------------------------------------------

        (
            deduplicated_df,
            duplicate_df
        ) = deduplicate_valid_records(
            valid_df
        )

        deduplicated_df.persist(
            StorageLevel.MEMORY_AND_DISK
        )

        duplicate_df.persist(
            StorageLevel.MEMORY_AND_DISK
        )

        valid_after_dedup_count = (
            deduplicated_df.count()
        )

        duplicate_count = (
            duplicate_df.count()
        )

        print(
            "Duplicates removed:",
            duplicate_count
        )

        print(
            "Valid after deduplication:",
            valid_after_dedup_count
        )

        # ----------------------------------------------------
        # VERIFY DEDUPLICATION COUNTS
        # ----------------------------------------------------

        if (
            valid_before_dedup_count
            != valid_after_dedup_count
            + duplicate_count
        ):
            raise ValueError(
                "Deduplication reconciliation failed: "
                f"{valid_before_dedup_count} != "
                f"{valid_after_dedup_count} + "
                f"{duplicate_count}"
            )

        # ----------------------------------------------------
        # ICEBERG SILVER MERGE
        # ----------------------------------------------------

        if valid_after_dedup_count > 0:
            print(
                "\nMerging valid records into Silver..."
            )

            merge_sales_order_lines_to_silver(
                batch_spark,
                deduplicated_df,
                silver_table=SILVER_TABLE,
                stream_batch_id=stream_batch_id,
            )

            # Refresh cached catalog metadata after the commit.
            batch_spark.catalog.refreshTable(
                SILVER_TABLE
            )

            print(
                "Silver MERGE completed."
            )

            # ------------------------------------------------
            # POST-MERGE KEY VERIFICATION
            # ------------------------------------------------

            incoming_keys_found_in_silver = (
                count_incoming_keys_found_in_silver(
                    batch_spark,
                    deduplicated_df,
                    silver_table=SILVER_TABLE,
                )
            )

            incoming_versions_satisfied = (
                count_incoming_versions_satisfied_in_silver(
                    batch_spark,
                    deduplicated_df,
                    silver_table=SILVER_TABLE,
                )
            )

        else:
            incoming_keys_found_in_silver = 0
            incoming_versions_satisfied = 0

            print(
                "No valid records available for Silver MERGE."
            )

        print(
            "Incoming keys found in Silver:",
            incoming_keys_found_in_silver
        )

        print(
            "Incoming versions satisfied in Silver:",
            incoming_versions_satisfied
        )

        # ----------------------------------------------------
        # FINAL RECONCILIATION
        # ----------------------------------------------------

        reconciliation_details = (
            evaluate_micro_batch_reconciliation(
                source_count=source_count,
                valid_before_dedup_count=(
                    valid_before_dedup_count
                ),
                rejected_count=rejected_count,
                duplicate_count=duplicate_count,
                valid_after_dedup_count=(
                    valid_after_dedup_count
                ),
                incoming_keys_found_in_silver=(
                    incoming_keys_found_in_silver
                ),
            )
        )

        reconciliation_status = (
            reconciliation_details[
                "reconciliation_status"
            ]
        )

        # The version check is stronger than key existence.
        version_reconciliation_passed = (
            incoming_versions_satisfied
            == valid_after_dedup_count
        )

        if not version_reconciliation_passed:
            reconciliation_status = "FAILED"

        print(
            "\nReconciliation details:",
            reconciliation_details
        )

        print(
            "Version reconciliation passed:",
            version_reconciliation_passed
        )

        # ----------------------------------------------------
        # WRITE SUCCESS/FAILED RECONCILIATION AUDIT
        # ----------------------------------------------------

        final_status = (
            "SUCCESS"
            if reconciliation_status
            == "RECONCILED"
            else "FAILED"
        )

        completed_audit_row = (
            create_silver_audit_row(
                pipeline_run_id=PIPELINE_RUN_ID,
                stream_batch_id=stream_batch_id,
                status=final_status,
                source_count=source_count,
                valid_before_dedup_count=(
                    valid_before_dedup_count
                ),
                rejected_count=rejected_count,
                duplicate_count=duplicate_count,
                valid_after_dedup_count=(
                    valid_after_dedup_count
                ),
                incoming_keys_found_in_silver=(
                    incoming_keys_found_in_silver
                ),
                reconciliation_status=(
                    reconciliation_status
                ),
                exploration_status=(
                    exploration_status
                ),
                expectation_status=(
                    expectation_status
                ),
                error_message=None,
                started_at=batch_started_at,
                completed_at=datetime.utcnow(),
            )
        )

        write_silver_audit(
            batch_spark,
            completed_audit_row,
            audit_table=SILVER_AUDIT_TABLE,
        )

        if reconciliation_status != "RECONCILED":
            raise ValueError(
                "The Bronze-to-Silver micro-batch "
                "failed reconciliation."
            )

        print(
            "\nMicro-batch completed successfully."
        )

    except Exception as processing_error:
        # ----------------------------------------------------
        # WRITE FAILURE AUDIT
        #
        # Avoid replacing the original exception if audit
        # writing also fails.
        # ----------------------------------------------------

        print(
            "\nMicro-batch processing failed:",
            processing_error
        )

        failure_audit_row = create_silver_audit_row(
            pipeline_run_id=PIPELINE_RUN_ID,
            stream_batch_id=stream_batch_id,
            status="FAILED",
            source_count=source_count,
            valid_before_dedup_count=(
                valid_before_dedup_count
            ),
            rejected_count=rejected_count,
            duplicate_count=duplicate_count,
            valid_after_dedup_count=(
                valid_after_dedup_count
            ),
            incoming_keys_found_in_silver=(
                incoming_keys_found_in_silver
            ),
            reconciliation_status=(
                reconciliation_status
            ),
            exploration_status=(
                exploration_status
            ),
            expectation_status=(
                expectation_status
            ),
            error_message=(
                f"{type(processing_error).__name__}: "
                f"{processing_error}"
            ),
            started_at=batch_started_at,
            completed_at=datetime.utcnow(),
        )

        try:
            write_silver_audit(
                batch_df.sparkSession,
                failure_audit_row,
                audit_table=SILVER_AUDIT_TABLE,
            )

        except Exception as audit_error:
            print(
                "Failure audit could not be written:",
                audit_error
            )

        # Rethrow so Spark marks the micro-batch as failed and
        # does not advance the checkpoint.
        raise

    finally:
        # ----------------------------------------------------
        # RELEASE ONLY CALLBACK-OWNED CACHES
        # ----------------------------------------------------

        if duplicate_df is not None:
            duplicate_df.unpersist()

        if deduplicated_df is not None:
            deduplicated_df.unpersist()

        if rejected_df is not None:
            rejected_df.unpersist()

        if valid_df is not None:
            valid_df.unpersist()

        if transformed_df is not None:
            transformed_df.unpersist()

        batch_df.unpersist()

        print(
            "Micro-batch caches released."
        )

Pipeline run ID: sales-order-lines-b047a944-e70a-4c9d-ac83-af39a081b864


In [21]:
# ============================================================
# CREATE BRONZE STREAMING DATAFRAME
#
# Only business_schema is supplied because the CSV files
# physically contain only the 11 business columns.
#
# batch_id, write_date_partition, and source-file lineage are
# derived inside the foreachBatch callback.
# ============================================================

bronze_stream_df = (
    spark.readStream
    .format("csv")
    .schema(business_schema)

    # CSV parsing options
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")

    # Bronze timestamps were written in this format:
    # 2026-10-07T17:50:27.000Z
    .option(
        "timestampFormat",
        BRONZE_TIMESTAMP_FORMAT
    )

    # Read files under all nested batch and date directories
    .option(
        "recursiveFileLookup",
        "true"
    )

    # Bronze files were generated by Spark and each record is
    # stored on one physical line.
    .option(
        "multiLine",
        "false"
    )

    # Process a limited number of files per micro-batch.
    # This still allows availableNow to process all currently
    # available files across multiple micro-batches.
    .option(
        "maxFilesPerTrigger",
        "5"
    )

    .load(BRONZE_PATH)
)


# ============================================================
# VERIFY STREAMING DATAFRAME
# ============================================================

print(
    "Is streaming DataFrame:",
    bronze_stream_df.isStreaming
)

print(
    "\nBronze streaming schema:"
)

bronze_stream_df.printSchema()


# ============================================================
# VERIFY EXPECTED COLUMN NAMES
# ============================================================

expected_stream_columns = [
    field.name
    for field in business_schema.fields
]

actual_stream_columns = (
    bronze_stream_df.columns
)

print(
    "\nExpected stream columns:",
    expected_stream_columns
)

print(
    "\nActual stream columns:",
    actual_stream_columns
)

if actual_stream_columns != expected_stream_columns:
    raise ValueError(
        "Bronze streaming columns do not match "
        "the business schema.\n"
        f"Expected: {expected_stream_columns}\n"
        f"Actual:   {actual_stream_columns}"
    )

print(
    "\nStreaming schema validation: PASSED"
)


# ============================================================
# VERIFY NO QUERY HAS STARTED YET
# ============================================================

active_query_names = [
    query.name
    for query in spark.streams.active
]

print(
    "\nCurrently active streaming queries:",
    active_query_names
)

print(
    "\nBronze streaming DataFrame created successfully. "
    "The stream has not been started yet."
)

Is streaming DataFrame: True

Bronze streaming schema:
root
 |-- id: long (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_uom_qty: double (nullable = true)
 |-- price_unit: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- price_subtotal: double (nullable = true)
 |-- create_date: timestamp (nullable = true)
 |-- write_date: timestamp (nullable = true)
 |-- qty_delivered: double (nullable = true)
 |-- qty_invoiced: double (nullable = true)


Expected stream columns: ['id', 'order_id', 'product_id', 'product_uom_qty', 'price_unit', 'discount', 'price_subtotal', 'create_date', 'write_date', 'qty_delivered', 'qty_invoiced']

Actual stream columns: ['id', 'order_id', 'product_id', 'product_uom_qty', 'price_unit', 'discount', 'price_subtotal', 'create_date', 'write_date', 'qty_delivered', 'qty_invoiced']

Streaming schema validation: PASSED

Currently active streaming queries: []

Bronze streaming DataFrame

In [23]:
# ============================================================
# FIX CHECKPOINT PATH WITH EXPLICIT HDFS AUTHORITY
# ============================================================

HDFS_DEFAULT_FS = (
    spark.sparkContext
    ._jsc
    .hadoopConfiguration()
    .get("fs.defaultFS")
)

print(
    "Configured fs.defaultFS:",
    HDFS_DEFAULT_FS
)


# Use the explicit NameNode authority observed in HDFS paths.
CHECKPOINT_PATH = (
    "hdfs://localhost:9000/"
    "capstone/checkpoints/"
    "bronze_to_silver/"
    "sales_order_lines"
)

print(
    "Corrected checkpoint path:",
    CHECKPOINT_PATH
)


# ============================================================
# VERIFY CHECKPOINT FILESYSTEM
# ============================================================

checkpoint_hadoop_path = (
    spark._jvm.org.apache.hadoop.fs.Path(
        CHECKPOINT_PATH
    )
)

checkpoint_fs = (
    checkpoint_hadoop_path.getFileSystem(
        spark._jsc.hadoopConfiguration()
    )
)

print(
    "Checkpoint filesystem URI:",
    checkpoint_fs.getUri().toString()
)

print(
    "Checkpoint path exists:",
    checkpoint_fs.exists(
        checkpoint_hadoop_path
    )
)

Configured fs.defaultFS: hdfs://localhost:9000
Corrected checkpoint path: hdfs://localhost:9000/capstone/checkpoints/bronze_to_silver/sales_order_lines
Checkpoint filesystem URI: hdfs://localhost:9000
Checkpoint path exists: False


In [30]:
# ============================================================
# START BRONZE-TO-SILVER STREAM
# ============================================================

STREAM_QUERY_NAME = (
    "sales_order_lines_bronze_to_silver"
)


# ============================================================
# PREVENT DUPLICATE ACTIVE QUERY
# ============================================================

existing_query = next(
    (
        query
        for query in spark.streams.active
        if query.name == STREAM_QUERY_NAME
    ),
    None
)

if existing_query is not None:
    raise RuntimeError(
        f"A streaming query named '{STREAM_QUERY_NAME}' "
        "is already active."
    )


# ============================================================
# DISPLAY EXECUTION DETAILS
# ============================================================

print("Starting Bronze-to-Silver stream")
print("-" * 70)

print(
    "Query name:",
    STREAM_QUERY_NAME
)

print(
    "Pipeline run ID:",
    PIPELINE_RUN_ID
)

print(
    "Bronze path:",
    BRONZE_PATH
)

print(
    "Checkpoint path:",
    CHECKPOINT_PATH
)

print(
    "Trigger:",
    "availableNow=True"
)

print(
    "Silver table:",
    SILVER_TABLE
)

print(
    "Rejected table:",
    REJECTED_TABLE
)

print(
    "Audit table:",
    SILVER_AUDIT_TABLE
)


# ============================================================
# START QUERY
# ============================================================

bronze_to_silver_query = (
    bronze_stream_df
    .writeStream

    .queryName(
        STREAM_QUERY_NAME
    )

    .foreachBatch(
        process_sales_order_lines_batch
    )

    .option(
        "checkpointLocation",
        CHECKPOINT_PATH
    )

    .trigger(
        availableNow=True
    )

    .start()
)


# ============================================================
# DISPLAY QUERY IDENTIFIERS
# ============================================================

print("\nStreaming query started.")

print(
    "Query ID:",
    bronze_to_silver_query.id
)

print(
    "Query run ID:",
    bronze_to_silver_query.runId
)

print(
    "Query active:",
    bronze_to_silver_query.isActive
)


# ============================================================
# WAIT FOR AVAILABLE DATA TO FINISH PROCESSING
# ============================================================

print(
    "\nWaiting for all currently available "
    "Bronze files to be processed..."
)

bronze_to_silver_query.awaitTermination()


# ============================================================
# DISPLAY FINAL QUERY STATUS
# ============================================================

print("\nStreaming query completed.")

print(
    "Query active:",
    bronze_to_silver_query.isActive
)

print(
    "Final query status:",
    bronze_to_silver_query.status
)

print(
    "Query exception:",
    bronze_to_silver_query.exception()
)

Starting Bronze-to-Silver stream
----------------------------------------------------------------------
Query name: sales_order_lines_bronze_to_silver
Pipeline run ID: sales-order-lines-b047a944-e70a-4c9d-ac83-af39a081b864
Bronze path: hdfs:///capstone/warehouse/bronze/sales_order_lines
Checkpoint path: hdfs://localhost:9000/capstone/checkpoints/bronze_to_silver/sales_order_lines
Trigger: availableNow=True
Silver table: datalake.silver.sales_order_lines
Rejected table: datalake.quarantine.sales_order_lines_rejected
Audit table: datalake.operations.silver_ingestion_audit

Streaming query started.
Query ID: 36c9e456-a09f-4d86-a005-980f8da38a91
Query run ID: dd34b7a6-1570-47ff-aae3-7a88f824e091
Query active: True

Waiting for all currently available Bronze files to be processed...

Processing stream batch: 0
Batch run ID: sales-order-lines-b047a944-e70a-4c9d-ac83-af39a081b864-batch-0


26/10/12 04:08:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Source records: 10497

Running data exploration...


Data exploration completed.

Running Great Expectations...


26/10/12 04:08:08 WARN CacheManager: Asked to cache already cached data.


Calculating Metrics:   0%|          | 0/76 [00:00<?, ?it/s]

Great Expectations profiling completed.

Transforming and validating records...
Valid before deduplication: 10491
Rejected records: 6
Rejected records written: 6
Duplicates removed: 6994
Valid after deduplication: 3497

Merging valid records into Silver...
Silver staging table: datalake.operations.sales_order_lines_staging_sales_order_lines_b047a944_e70a_4c9d_ac83_af39a081b864_0
Expected staging count: 3497
Written staging count: 3497
Silver MERGE completed from staging table.
Staging table removed: datalake.operations.sales_order_lines_staging_sales_order_lines_b047a944_e70a_4c9d_ac83_af39a081b864_0
Silver MERGE completed.
Incoming keys found in Silver: 3497
Incoming versions satisfied in Silver: 3497

Reconciliation details: {'source_split_passed': True, 'deduplication_passed': True, 'silver_merge_passed': True, 'reconciliation_status': 'RECONCILED'}
Version reconciliation passed: True
Silver audit record written: b2e53d64-4949-4933-85a2-23109aee16dc

Micro-batch completed successful

In [35]:
# ============================================================
# REFRESH ICEBERG TABLE METADATA
# ============================================================

for table_name in [
    SILVER_TABLE,
    REJECTED_TABLE,
    SILVER_AUDIT_TABLE,
    EXPLORATION_TABLE,
    EXPECTATION_TABLE,
]:
    if spark.catalog.tableExists(table_name):
        spark.catalog.refreshTable(table_name)


# ============================================================
# VALIDATE SILVER TABLE
# ============================================================

silver_df = spark.table(SILVER_TABLE)

silver_count = silver_df.count()

silver_distinct_id_count = (
    silver_df
    .select("sales_order_line_id")
    .distinct()
    .count()
)

silver_duplicate_id_count = (
    silver_df
    .groupBy("sales_order_line_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

silver_null_id_count = (
    silver_df
    .filter(
        F.col("sales_order_line_id").isNull()
    )
    .count()
)


print("Silver table validation")
print("-" * 70)

print(
    "Silver record count:",
    silver_count
)

print(
    "Distinct sales_order_line_id count:",
    silver_distinct_id_count
)

print(
    "Duplicate ID count:",
    silver_duplicate_id_count
)

print(
    "Null ID count:",
    silver_null_id_count
)


if silver_count != silver_distinct_id_count:
    raise ValueError(
        "Silver primary-key reconciliation failed: "
        f"record_count={silver_count}, "
        f"distinct_id_count={silver_distinct_id_count}"
    )

if silver_duplicate_id_count > 0:
    raise ValueError(
        "Duplicate sales_order_line_id values exist in Silver."
    )

if silver_null_id_count > 0:
    raise ValueError(
        "Null sales_order_line_id values exist in Silver."
    )

print(
    "Silver primary-key validation: PASSED"
)


# ============================================================
# DISPLAY SILVER PREVIEW
# ============================================================

print("\nSilver data preview:")

silver_df.select(
    "sales_order_line_id",
    "sales_order_id",
    "sales_order_name",
    "product_id",
    "product_display_name",
    "product_uom_qty",
    "price_unit",
    "discount",
    "price_subtotal",
    "create_date",
    "write_date",
    "source_batch_id",
    "stream_batch_id"
).orderBy(
    "sales_order_line_id"
).show(
    5,
    truncate=False
)


# ============================================================
# DISPLAY SILVER RECORD DISTRIBUTION
# ============================================================

print("\nSilver records by source batch:")

silver_df.groupBy(
    "source_batch_id"
).agg(
    F.count("*").alias("record_count"),
    F.min("write_date").alias(
        "minimum_write_date"
    ),
    F.max("write_date").alias(
        "maximum_write_date"
    )
).orderBy(
    "source_batch_id"
).show(
    truncate=False
)


print("\nSilver records by write date:")

silver_df.groupBy(
    F.to_date("write_date").alias(
        "write_date_partition"
    )
).count().orderBy(
    "write_date_partition"
).show(
    truncate=False
)


# ============================================================
# VALIDATE REJECTED RECORDS FOR THIS PIPELINE RUN
# ============================================================

current_run_rejected_df = (
    spark.table(REJECTED_TABLE)
    .filter(
        F.col("pipeline_run_id")
        == PIPELINE_RUN_ID
    )
)

current_run_rejected_count = (
    current_run_rejected_df.count()
)


print("\nRejected-record validation")
print("-" * 70)

print(
    "Rejected records for current run:",
    current_run_rejected_count
)


print("\nRejected records by failed rule:")

(
    current_run_rejected_df
    .select(
        F.explode(
            "failed_rule_names"
        ).alias("failed_rule_name")
    )
    .groupBy(
        "failed_rule_name"
    )
    .count()
    .orderBy(
        F.col("count").desc(),
        F.col("failed_rule_name")
    )
    .show(
        truncate=False
    )
)


print("\nRejected-record details:")

current_run_rejected_df.select(
    "id",
    "source_batch_id",
    "failed_rule_names",
    "rejection_reasons",
    "source_file",
    "stream_batch_id",
    "rejected_at"
).orderBy(
    "source_batch_id",
    "id"
).show(
    5,
    truncate=False,
    vertical=True
)


# ============================================================
# VALIDATE MICRO-BATCH AUDIT
# ============================================================

current_run_audit_df = (
    spark.table(SILVER_AUDIT_TABLE)
    .filter(
        F.col("pipeline_run_id")
        == PIPELINE_RUN_ID
    )
    .orderBy(
        "stream_batch_id",
        "started_at"
    )
)


print("\nSilver ingestion audit")
print("-" * 70)

current_run_audit_df.select(
    "pipeline_run_id",
    "stream_batch_id",
    "status",
    "source_count",
    "valid_before_dedup_count",
    "rejected_count",
    "duplicate_count",
    "valid_after_dedup_count",
    "incoming_keys_found_in_silver",
    "reconciliation_status",
    "exploration_status",
    "expectation_status",
    "error_message",
    "started_at",
    "completed_at"
).show(
    truncate=False,
    vertical=True
)


failed_audit_count = (
    current_run_audit_df
    .filter(
        (F.col("status") != "SUCCESS")
        | (
            F.col("reconciliation_status")
            != "RECONCILED"
        )
    )
    .count()
)

if failed_audit_count > 0:
    raise ValueError(
        "One or more micro-batches failed processing "
        "or reconciliation."
    )

print(
    "Silver audit validation: PASSED"
)


# ============================================================
# VALIDATE EXPECTED CURRENT-RUN COUNTS
# ============================================================

audit_totals = (
    current_run_audit_df
    .agg(
        F.sum("source_count").alias(
            "source_count"
        ),
        F.sum(
            "valid_before_dedup_count"
        ).alias(
            "valid_before_dedup_count"
        ),
        F.sum("rejected_count").alias(
            "rejected_count"
        ),
        F.sum("duplicate_count").alias(
            "duplicate_count"
        ),
        F.sum(
            "valid_after_dedup_count"
        ).alias(
            "valid_after_dedup_count"
        ),
    )
    .first()
)


print("\nCurrent-run totals")
print("-" * 70)

print(
    "Source count:",
    audit_totals["source_count"]
)

print(
    "Valid before deduplication:",
    audit_totals[
        "valid_before_dedup_count"
    ]
)

print(
    "Rejected count:",
    audit_totals["rejected_count"]
)

print(
    "Duplicate count:",
    audit_totals["duplicate_count"]
)

print(
    "Valid after deduplication:",
    audit_totals[
        "valid_after_dedup_count"
    ]
)


if (
    audit_totals["source_count"]
    !=
    audit_totals["valid_before_dedup_count"]
    + audit_totals["rejected_count"]
):
    raise ValueError(
        "Current-run source split reconciliation failed."
    )

if (
    audit_totals["valid_before_dedup_count"]
    !=
    audit_totals["valid_after_dedup_count"]
    + audit_totals["duplicate_count"]
):
    raise ValueError(
        "Current-run deduplication reconciliation failed."
    )

if (
    current_run_rejected_count
    != audit_totals["rejected_count"]
):
    raise ValueError(
        "Rejected-table count does not match "
        "the processing-audit count."
    )

print(
    "Current-run count reconciliation: PASSED"
)


# ============================================================
# INSPECT DATA-EXPLORATION RESULTS
#
# Filtering uses batch run ID because the callback passed:
# <pipeline_run_id>-batch-<stream_batch_id>
# ============================================================

print("\nData-exploration results")
print("-" * 70)

exploration_run_prefix = (
    PIPELINE_RUN_ID + "-batch-"
)

current_run_exploration_df = (
    spark.table(EXPLORATION_TABLE)
    .filter(
        F.col("run_id").startswith(
            exploration_run_prefix
        )
    )
)

print(
    "Exploration result rows:",
    current_run_exploration_df.count()
)

current_run_exploration_df.show(
    5,
    truncate=False
)


# ============================================================
# INSPECT GREAT EXPECTATIONS RESULTS
# ============================================================

print("\nGreat Expectations results")
print("-" * 70)

current_run_expectation_df = (
    spark.table(EXPECTATION_TABLE)
    .filter(
        F.col("run_id").startswith(
            exploration_run_prefix
        )
    )
)

print(
    "Expectation result rows:",
    current_run_expectation_df.count()
)

current_run_expectation_df.show(
    5,
    truncate=False
)

Silver table validation
----------------------------------------------------------------------
Silver record count: 3497
Distinct sales_order_line_id count: 3497
Duplicate ID count: 0
Null ID count: 0
Silver primary-key validation: PASSED

Silver data preview:
+-------------------+--------------+----------------+----------+--------------------------------------------+---------------+----------+--------+--------------+-------------------+-------------------+---------------+---------------+
|sales_order_line_id|sales_order_id|sales_order_name|product_id|product_display_name                        |product_uom_qty|price_unit|discount|price_subtotal|create_date        |write_date         |source_batch_id|stream_batch_id|
+-------------------+--------------+----------------+----------+--------------------------------------------+---------------+----------+--------+--------------+-------------------+-------------------+---------------+---------------+
|1                  |14            |S000

In [36]:
# ============================================================
# CAPTURE COUNTS BEFORE CHECKPOINT TEST
# ============================================================

silver_count_before = (
    spark.table(SILVER_TABLE).count()
)

rejected_count_before = (
    spark.table(REJECTED_TABLE).count()
)

audit_count_before = (
    spark.table(SILVER_AUDIT_TABLE).count()
)


print("Counts before checkpoint test")
print("-" * 70)

print(
    "Silver count:",
    silver_count_before
)

print(
    "Rejected count:",
    rejected_count_before
)

print(
    "Silver audit count:",
    audit_count_before
)


# ============================================================
# START THE STREAM AGAIN WITH THE SAME CHECKPOINT
#
# Since no new Bronze files have been added, Spark should not
# send any records to foreachBatch.
# ============================================================

CHECKPOINT_TEST_QUERY_NAME = (
    "sales_order_lines_checkpoint_test"
)

checkpoint_test_query = (
    bronze_stream_df
    .writeStream

    .queryName(
        CHECKPOINT_TEST_QUERY_NAME
    )

    .foreachBatch(
        process_sales_order_lines_batch
    )

    .option(
        "checkpointLocation",
        CHECKPOINT_PATH
    )

    .trigger(
        availableNow=True
    )

    .start()
)


print("\nCheckpoint test query started.")

print(
    "Query ID:",
    checkpoint_test_query.id
)

print(
    "Query run ID:",
    checkpoint_test_query.runId
)


# ============================================================
# WAIT FOR THE QUERY TO FINISH
# ============================================================

checkpoint_test_query.awaitTermination()


print("\nCheckpoint test query completed.")

print(
    "Query active:",
    checkpoint_test_query.isActive
)

print(
    "Query exception:",
    checkpoint_test_query.exception()
)


# ============================================================
# INSPECT STREAMING PROGRESS
# ============================================================

checkpoint_progress = (
    checkpoint_test_query.recentProgress
)

print(
    "\nNumber of progress events:",
    len(checkpoint_progress)
)

total_checkpoint_input_rows = sum(
    progress.get(
        "numInputRows",
        0
    )
    for progress in checkpoint_progress
)

print(
    "Total input rows during checkpoint test:",
    total_checkpoint_input_rows
)


if checkpoint_progress:
    print("\nCheckpoint query progress:")

    for progress in checkpoint_progress:
        print(
            json.dumps(
                progress,
                indent=4,
                default=str
            )
        )
else:
    print(
        "No micro-batch progress events were generated. "
        "No new files were available."
    )


# ============================================================
# REFRESH TABLE METADATA
# ============================================================

spark.catalog.refreshTable(
    SILVER_TABLE
)

spark.catalog.refreshTable(
    REJECTED_TABLE
)

spark.catalog.refreshTable(
    SILVER_AUDIT_TABLE
)


# ============================================================
# CAPTURE COUNTS AFTER CHECKPOINT TEST
# ============================================================

silver_count_after = (
    spark.table(SILVER_TABLE).count()
)

rejected_count_after = (
    spark.table(REJECTED_TABLE).count()
)

audit_count_after = (
    spark.table(SILVER_AUDIT_TABLE).count()
)


print("\nCounts after checkpoint test")
print("-" * 70)

print(
    "Silver count:",
    silver_count_after
)

print(
    "Rejected count:",
    rejected_count_after
)

print(
    "Silver audit count:",
    audit_count_after
)


# ============================================================
# VERIFY THAT NO DATA WAS REPROCESSED
# ============================================================

silver_unchanged = (
    silver_count_after
    == silver_count_before
)

rejected_unchanged = (
    rejected_count_after
    == rejected_count_before
)

audit_unchanged = (
    audit_count_after
    == audit_count_before
)

no_input_processed = (
    total_checkpoint_input_rows == 0
)


print("\nCheckpoint validation")
print("-" * 70)

print(
    "No input rows processed:",
    no_input_processed
)

print(
    "Silver count unchanged:",
    silver_unchanged
)

print(
    "Rejected count unchanged:",
    rejected_unchanged
)

print(
    "Audit count unchanged:",
    audit_unchanged
)


if not no_input_processed:
    raise ValueError(
        "The checkpoint test unexpectedly processed "
        f"{total_checkpoint_input_rows} input rows."
    )

if not silver_unchanged:
    raise ValueError(
        "Silver count changed during the checkpoint test."
    )

if not rejected_unchanged:
    raise ValueError(
        "Rejected-record count changed during the "
        "checkpoint test."
    )

if not audit_unchanged:
    raise ValueError(
        "Silver audit count changed during the "
        "checkpoint test."
    )


print(
    "\nCheckpoint and idempotency validation: PASSED"
)

Counts before checkpoint test
----------------------------------------------------------------------
Silver count: 3497
Rejected count: 6
Silver audit count: 1

Checkpoint test query started.
Query ID: 36c9e456-a09f-4d86-a005-980f8da38a91
Query run ID: 602e0874-63cf-4a17-b006-d1dd8e3083a2

Checkpoint test query completed.
Query active: False
Query exception: None

Number of progress events: 1
Total input rows during checkpoint test: 0

Checkpoint query progress:
{
    "id": "36c9e456-a09f-4d86-a005-980f8da38a91",
    "runId": "602e0874-63cf-4a17-b006-d1dd8e3083a2",
    "name": "sales_order_lines_checkpoint_test",
    "timestamp": "2026-10-12T04:16:10.830Z",
    "batchId": 1,
    "numInputRows": 0,
    "inputRowsPerSecond": 0.0,
    "processedRowsPerSecond": 0.0,
    "durationMs": {
        "latestOffset": 0,
        "triggerExecution": 28
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "FileStreamSource[hdfs://localhost:9000/capstone/warehouse/bro

26/10/12 04:16:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.



Counts after checkpoint test
----------------------------------------------------------------------
Silver count: 3497
Rejected count: 6
Silver audit count: 1

Checkpoint validation
----------------------------------------------------------------------
No input rows processed: True
Silver count unchanged: True
Rejected count unchanged: True
Audit count unchanged: True

Checkpoint and idempotency validation: PASSED
